# 📚 Cuaderno de Ejercicios: Análisis Semántico
## Compiladores — Ejercicios Teórico-Prácticos

---

Este cuaderno contiene **40 ejercicios** (20 por unidad) que combinan teoría y práctica para comprender a fondo el Análisis Semántico en compiladores.

| | Unidad | Tema |
|---|---|---|
| 🔵 | **Unidad 1** | Traducción Dirigida por la Sintaxis (TDS) |
| 🟢 | **Unidad 2** | Declaración y Comprobación de Tipos |

**Instrucciones:** Cada ejercicio tiene una sección de explicación, una celda de código que puedes ejecutar, y una pregunta de reflexión o tarea.

---
# 🔵 UNIDAD 1: Traducción Dirigida por la Sintaxis
---

## Ejercicio U1-01 (Teórico) — ¿Qué es una TDS?

Una **Traducción Dirigida por la Sintaxis (TDS)** es un método para asociar significado a las construcciones de un lenguaje de programación usando una gramática libre del contexto como estructura base.

Cada símbolo gramatical puede tener **atributos** (propiedades que almacenan información) y cada producción tiene **reglas semánticas** que calculan esos atributos.

### Ejemplo conceptual:
```
Producción:       E → E₁ + T
Regla semántica:  E.val = E₁.val + T.val
```

El código siguiente simula la estructura de una TDS simple.

In [ ]:
# Ejercicio U1-01: Estructura básica de una TDS
# Una TDS asocia producciones con reglas semánticas.
# Aquí representamos la TDS como un diccionario de producciones → reglas.

TDS_calculadora = {
    # Producción: (Símbolo izquierdo, símbolos derechos) -> regla semántica como función
    ('E', ('E', '+', 'T')): lambda vals: vals['E'] + vals['T'],
    ('E', ('E', '-', 'T')): lambda vals: vals['E'] - vals['T'],
    ('E', ('T',)):          lambda vals: vals['T'],
    ('T', ('T', '*', 'F')): lambda vals: vals['T'] * vals['F'],
    ('T', ('F',)):          lambda vals: vals['F'],
    ('F', ('num',)):        lambda vals: vals['num'],
}

# Simulamos la evaluación de 3 + 4 * 2
# Árbol de análisis: E -> E + T -> T + T -> F + T*F -> num + num*num
F1 = TDS_calculadora[('F', ('num',))]({'num': 3})         # F -> num (3)
F2 = TDS_calculadora[('F', ('num',))]({'num': 4})         # F -> num (4)
F3 = TDS_calculadora[('F', ('num',))]({'num': 2})         # F -> num (2)
T1 = TDS_calculadora[('T', ('F',))]({'F': F1})            # T -> F (3)
T2 = TDS_calculadora[('T', ('T', '*', 'F'))]({'T': F2, 'F': F3})  # T -> T*F (8)
E1 = TDS_calculadora[('E', ('T',))]({'T': T1})            # E -> T (3)
E2 = TDS_calculadora[('E', ('E', '+', 'T'))]({'E': E1, 'T': T2})  # E -> E+T

print(f"Resultado de 3 + 4 * 2 = {E2}")
print(f"\nProducciones de la TDS definidas: {len(TDS_calculadora)}")
for prod in TDS_calculadora:
    print(f"  {prod[0]} -> {' '.join(prod[1])}")

**📝 Reflexión:** ¿Por qué la TDS respeta la precedencia de operadores (`*` antes de `+`)? ¿Dónde está codificada esa precedencia?

---
## Ejercicio U1-02 (Práctico) — Árbol de Análisis Anotado

Un **árbol de análisis anotado** muestra los valores de los atributos en cada nodo.
Implementamos una clase para crear y visualizar estos árboles.

In [ ]:
# Ejercicio U1-02: Árbol de análisis anotado

class NodoArbol:
    def __init__(self, simbolo, hijos=None, atributos=None):
        self.simbolo = simbolo
        self.hijos = hijos or []
        self.atributos = atributos or {}

    def __repr__(self):
        attrs = ', '.join(f"{k}={v}" for k, v in self.atributos.items())
        return f"{self.simbolo}[{attrs}]"

    def imprimir(self, nivel=0):
        prefijo = "  " * nivel
        attrs = ', '.join(f"{k}={v}" for k, v in self.atributos.items())
        print(f"{prefijo}{'└─' if nivel>0 else ''}{self.simbolo} {{{attrs}}}")
        for hijo in self.hijos:
            hijo.imprimir(nivel + 1)

# Construimos el árbol anotado para la expresión: 5 + 3
# E -> E1 + T, con E1 -> T -> F -> num(5), T -> F -> num(3)
num5 = NodoArbol('num', atributos={'lexval': 5})
num3 = NodoArbol('num', atributos={'lexval': 3})

F_izq = NodoArbol('F', hijos=[num5], atributos={'val': 5})
F_der = NodoArbol('F', hijos=[num3], atributos={'val': 3})

T_izq = NodoArbol('T', hijos=[F_izq], atributos={'val': 5})
T_der = NodoArbol('T', hijos=[F_der], atributos={'val': 3})

E_izq = NodoArbol('E', hijos=[T_izq], atributos={'val': 5})
plus  = NodoArbol('+', atributos={})
E_raiz = NodoArbol('E', hijos=[E_izq, plus, T_der], atributos={'val': 5 + 3})

print("Árbol de análisis anotado para: 5 + 3")
print("="*40)
E_raiz.imprimir()
print(f"\nValor final de la expresión: {E_raiz.atributos['val']}")

**📝 Tarea:** Extiende el árbol para representar la expresión `5 + 3 * 2`. ¿Cómo cambia la estructura del árbol respecto a `5 + 3`?

---
## Ejercicio U1-03 (Teórico) — Atributos Sintetizados vs Heredados

| Tipo | Flujo de información | Calculado desde |
|------|---------------------|------------------|
| **Sintetizado** | Hojas → Raíz (↑) | Atributos de los hijos |
| **Heredado** | Raíz → Hojas (↓) | Atributos del padre y hermanos izquierdos |

El siguiente código ilustra ambos tipos en una misma gramática.

In [ ]:
# Ejercicio U1-03: Atributos sintetizados vs heredados
# Gramática para declaraciones: D -> T L ; con L -> L, id | id
# T.tipo es SINTETIZADO (sube desde T)
# L.tipo_h es HEREDADO (baja desde T hacia la lista L)

class TablaSimbolosSimple:
    def __init__(self):
        self.tabla = {}

    def insertar(self, nombre, tipo):
        self.tabla[nombre] = tipo
        print(f"  ✔ Insertar: {nombre} → tipo={tipo}")

    def mostrar(self):
        print("\n📋 Tabla de Símbolos:")
        for nombre, tipo in self.tabla.items():
            print(f"  {nombre:10} | {tipo}")

def procesar_declaracion(tipo_T, identificadores, tabla):
    """
    Simula el procesamiento de: T id1, id2, id3 ;
    - tipo_T es SINTETIZADO a partir del token T
    - tipo_h es HEREDADO por cada elemento de la lista L
    """
    print(f"Procesando declaración: {tipo_T} {', '.join(identificadores)} ;")
    print(f"  → T.tipo (sintetizado) = '{tipo_T}'")
    print(f"  → L.tipo_h (heredado)  = '{tipo_T}' (pasado desde T a L)")
    print()
    # El tipo heredado fluye de izquierda a derecha a través de la lista
    tipo_heredado = tipo_T  # ← atributo heredado
    for ident in identificadores:
        tabla.insertar(ident, tipo_heredado)

tabla = TablaSimbolosSimple()
procesar_declaracion('int',   ['x', 'y', 'z'], tabla)
procesar_declaracion('float', ['precio', 'iva'], tabla)
tabla.mostrar()

**📝 Reflexión:** ¿Por qué no podemos usar solo atributos sintetizados para procesar la declaración `int x, y, z;`? ¿Qué problema surge?

---
## Ejercicio U1-04 (Práctico) — Definición S-atribuida: Calculadora Completa

Una TDS es **S-atribuida** si usa *solo* atributos sintetizados. Es la forma más simple de implementar y es compatible de forma natural con analizadores ascendentes.

In [ ]:
# Ejercicio U1-04: Calculadora S-atribuida usando análisis descendente
# Implementamos un evaluador de expresiones con TDS S-atribuida

import re

class CalculadoraTDS:
    """Evaluador de expresiones: implementación S-atribuida.
       Gramática:
         E → T E'
         E' → + T E' | - T E' | ε
         T → F T'
         T' → * F T' | / F T' | ε
         F → num | ( E )
    """

    def __init__(self, texto):
        # Tokenizar
        self.tokens = re.findall(r'\d+\.?\d*|[+\-*/()]', texto.replace(' ', ''))
        self.pos = 0
        print(f"Tokens: {self.tokens}")

    def token_actual(self):
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None

    def consumir(self):
        tok = self.token_actual()
        self.pos += 1
        return tok

    def E(self):
        val = self.T()          # E.val ← T.val (sintetizado)
        return self.E_prima(val)

    def E_prima(self, val_heredado):
        if self.token_actual() == '+':
            self.consumir()
            t_val = self.T()
            return self.E_prima(val_heredado + t_val)  # sintetizado
        elif self.token_actual() == '-':
            self.consumir()
            t_val = self.T()
            return self.E_prima(val_heredado - t_val)
        return val_heredado   # ε

    def T(self):
        val = self.F()
        return self.T_prima(val)

    def T_prima(self, val_heredado):
        if self.token_actual() == '*':
            self.consumir()
            f_val = self.F()
            return self.T_prima(val_heredado * f_val)
        elif self.token_actual() == '/':
            self.consumir()
            f_val = self.F()
            return self.T_prima(val_heredado / f_val)
        return val_heredado

    def F(self):
        tok = self.token_actual()
        if tok == '(':
            self.consumir()
            val = self.E()
            self.consumir()  # ')'
            return val
        else:
            self.consumir()
            return float(tok) if '.' in tok else int(tok)

# Pruebas
expresiones = ["3 + 4 * 2", "(3 + 4) * 2", "10 / 2 - 1", "2 * 3 + 4 * 5"]
for expr in expresiones:
    calc = CalculadoraTDS(expr)
    resultado = calc.E()
    print(f"  {expr} = {resultado}")
    print()

**📝 Tarea:** Añade soporte para el operador módulo `%` a la calculadora. ¿En qué función debes hacer el cambio?

---
## Ejercicio U1-05 (Teórico) — Definiciones L-atribuidas

Una TDS es **L-atribuida** si cada atributo heredado de un símbolo Xⱼ depende solo de:
- Atributos de los símbolos a su **izquierda** (X₁,...,Xⱼ₋₁)
- Atributos **heredados** del padre A

Toda S-atribuida es también L-atribuida, pero no al revés.

In [ ]:
# Ejercicio U1-05: Clasificación de TDS (S-atribuida vs L-atribuida)

def clasificar_tds(nombre, producciones_reglas):
    """
    producciones_reglas: lista de (producción_str, tipo_atributo, descripción)
    tipo_atributo: 'sintetizado' o 'heredado_izquierda' o 'heredado_derecha'
    """
    tiene_heredados = False
    es_l_atribuida  = True

    print(f"\nAnálisis TDS: '{nombre}'")
    print("-" * 60)
    for prod, tipo, desc in producciones_reglas:
        if tipo == 'sintetizado':
            icono = '✅'
        elif tipo == 'heredado_izquierda':
            icono = '🔵'
            tiene_heredados = True
        else:  # heredado_derecha
            icono = '❌'
            tiene_heredados = True
            es_l_atribuida = False
        print(f"  {icono} {prod}")
        print(f"     → {desc} ({tipo})")

    print()
    if not tiene_heredados:
        print(f"  → Clasificación: S-atribuida ✔ (y también L-atribuida)")
    elif es_l_atribuida:
        print(f"  → Clasificación: L-atribuida ✔ (pero NO S-atribuida)")
    else:
        print(f"  → Clasificación: NI S ni L atribuida ✘ — orden inválido")

# Ejemplo 1: Calculadora (S-atribuida)
clasificar_tds("Calculadora", [
    ("E → E₁ + T", 'sintetizado', "E.val = E₁.val + T.val"),
    ("T → T₁ * F", 'sintetizado', "T.val = T₁.val * F.val"),
    ("F → num",    'sintetizado', "F.val = num.lexval"),
])

# Ejemplo 2: Declaraciones con tipo heredado (L-atribuida)
clasificar_tds("Declaraciones de variables", [
    ("D → T L",     'sintetizado',        "T.tipo calculado desde token"),
    ("D → T L",     'heredado_izquierda', "L.tipo_h = T.tipo (de hermano izquierdo)"),
    ("L → L₁, id", 'heredado_izquierda', "L₁.tipo_h = L.tipo_h (del padre)"),
])

# Ejemplo 3: Inválido (heredado de hermano derecho)
clasificar_tds("TDS inválida", [
    ("A → B C D", 'heredado_derecha', "B.x = C.y  (¡C está a la derecha de B!)"),
])

**📝 Reflexión:** ¿Por qué las definiciones L-atribuidas son compatibles con analizadores descendentes (top-down) pero no siempre con ascendentes (bottom-up)?

---
## Ejercicio U1-06 (Práctico) — Grafo de Dependencias

Un **grafo de dependencias** modela las relaciones entre atributos para una cadena concreta. Si el grafo tiene ciclos, la TDS no puede evaluarse.

In [ ]:
# Ejercicio U1-06: Grafo de dependencias y ordenación topológica

from collections import defaultdict, deque

class GrafoDependencias:
    def __init__(self):
        self.nodos = set()
        self.aristas = defaultdict(set)  # a -> {b} significa "b depende de a"
        self.entrada = defaultdict(int)  # grado de entrada

    def agregar_nodo(self, nodo):
        self.nodos.add(nodo)
        if nodo not in self.entrada:
            self.entrada[nodo] = 0

    def agregar_arista(self, desde, hacia):
        """'hacia' depende de 'desde': desde → hacia"""
        self.agregar_nodo(desde)
        self.agregar_nodo(hacia)
        if hacia not in self.aristas[desde]:
            self.aristas[desde].add(hacia)
            self.entrada[hacia] += 1

    def ordenacion_topologica(self):
        """Algoritmo de Kahn para ordenación topológica."""
        grado = dict(self.entrada)
        cola = deque([n for n in self.nodos if grado[n] == 0])
        orden = []
        while cola:
            nodo = cola.popleft()
            orden.append(nodo)
            for vecino in sorted(self.aristas[nodo]):
                grado[vecino] -= 1
                if grado[vecino] == 0:
                    cola.append(vecino)
        if len(orden) == len(self.nodos):
            return orden, True  # Sin ciclos
        return orden, False     # Hay ciclos

# Construimos el grafo para: E → E₁ + T con E.val = E₁.val + T.val
# Atributos del árbol para "1 + 2":
#   F1.val, T1.val, E1.val, F2.val, T2.val, E.val
g = GrafoDependencias()
# num1.lexval → F1.val → T1.val → E1.val
g.agregar_arista('num1.lexval', 'F1.val')
g.agregar_arista('F1.val', 'T1.val')
g.agregar_arista('T1.val', 'E1.val')
# num2.lexval → F2.val → T2.val
g.agregar_arista('num2.lexval', 'F2.val')
g.agregar_arista('F2.val', 'T2.val')
# E1.val + T2.val → E.val
g.agregar_arista('E1.val', 'E.val')
g.agregar_arista('T2.val', 'E.val')

orden, valido = g.ordenacion_topologica()
print("Grafo de dependencias para: num₁ + num₂")
print("Aristas (A → B = 'B depende de A'):")
for desde, hijos in g.aristas.items():
    for hijo in hijos:
        print(f"  {desde} ──▶ {hijo}")
print()
print(f"{'✅ Sin ciclos' if valido else '❌ HAY CICLOS — TDS inválida'}")
print(f"Orden de evaluación válido:")
for i, nodo in enumerate(orden, 1):
    print(f"  {i}. {nodo}")

**📝 Tarea:** Agrega una arista `E.val → num1.lexval` al grafo. ¿Qué pasa con la ordenación topológica? ¿Qué significa esto para la TDS?

---
## Ejercicio U1-07 (Práctico) — Esquemas de Traducción: Infijo → Postfijo

Un **esquema de traducción** coloca acciones semánticas `{...}` dentro de las producciones. La posición determina *cuándo* se ejecuta cada acción.

In [ ]:
# Ejercicio U1-07: Esquema de traducción — infijo a postfijo
# Gramática con acciones semánticas:
#   E → T E'
#   E' → + T {imprimir('+')} E' | ε
#   T → F T'
#   T' → * F {imprimir('*')} T' | ε
#   F → num {imprimir(num)}

import re

class TraductorPostfijo:
    def __init__(self, texto):
        self.tokens = re.findall(r'\d+|[+\-*/()]', texto.replace(' ', ''))
        self.pos = 0
        self.salida = []

    def token_actual(self):
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None

    def consumir(self):
        tok = self.token_actual(); self.pos += 1; return tok

    def emitir(self, s):
        self.salida.append(str(s))

    def traducir(self):
        self.E()
        return ' '.join(self.salida)

    def E(self):
        self.T()
        self.E_prima()

    def E_prima(self):
        if self.token_actual() == '+':
            self.consumir()
            self.T()
            self.emitir('+')   # {imprimir('+')} — DESPUÉS de procesar T
            self.E_prima()
        elif self.token_actual() == '-':
            self.consumir()
            self.T()
            self.emitir('-')
            self.E_prima()

    def T(self):
        self.F()
        self.T_prima()

    def T_prima(self):
        if self.token_actual() == '*':
            self.consumir()
            self.F()
            self.emitir('*')   # {imprimir('*')}
            self.T_prima()
        elif self.token_actual() == '/':
            self.consumir()
            self.F()
            self.emitir('/')
            self.T_prima()

    def F(self):
        tok = self.token_actual()
        if tok == '(':
            self.consumir()
            self.E()
            self.consumir()  # ')'
        else:
            num = self.consumir()
            self.emitir(num)   # {imprimir(num)}

# Pruebas
casos = [
    "3 + 4",
    "3 + 4 * 2",
    "(3 + 4) * 2",
    "9 - 3 + 1",
    "2 * 3 + 4 * 5 - 1",
]
print(f"{'Expresión infija':25} → Postfijo")
print("-" * 50)
for expr in casos:
    t = TraductorPostfijo(expr)
    postfijo = t.traducir()
    print(f"  {expr:23} → {postfijo}")

**📝 Tarea:** Modifica el traductor para que genere notación **prefija** (operador primero). Por ejemplo, `3 + 4` → `+ 3 4`. ¿Dónde debes mover las llamadas a `emitir()`?

---
## Ejercicio U1-08 (Teórico) — Comparación TDS vs Esquema de Traducción

Analicemos las diferencias clave entre ambos formalismos.

In [ ]:
# Ejercicio U1-08: TDS declarativa vs Esquema imperativo

comparacion = {
    'Característica': [
        'Tipo de especificación',
        'Orden de evaluación',
        'Posición de acciones',
        'Usado en herramientas',
        'Compatibilidad analizadores',
        'Verificación de ciclos',
    ],
    'TDS (Declarativa)': [
        'Declarativa — QUÉ calcular',
        'No especificado (se deduce del grafo)',
        'Asociada a la producción completa',
        'YACC/Bison (especificación)',
        'S-atribuida → ascendente; L-atribuida → descendente',
        'Requiere análisis del grafo de dependencias',
    ],
    'Esquema de traducción (Imperativo)': [
        'Imperativa — CÓMO y CUÁNDO calcular',
        'Explícito — posición de {} en producción',
        'Incrustada DENTRO del cuerpo de la producción',
        'YACC/Bison (implementación)',
        'L-atribuida → descendente (nativamente)',
        'Posible ciclo implícito si mal diseñado',
    ]
}

print(f"{'Característica':45} | {'TDS':35} | Esquema de traducción")
print("-" * 130)
for i in range(len(comparacion['Característica'])):
    c  = comparacion['Característica'][i]
    t  = comparacion['TDS (Declarativa)'][i]
    e  = comparacion['Esquema de traducción (Imperativo)'][i]
    print(f"  {c:43} | {t:33} | {e}")

print("\n" + "="*60)
print("EJEMPLO: Misma semántica, dos formalismos diferentes")
print("="*60)
print("\nComo TDS (declarativa):")
print("  Producción:      E → E₁ + T")
print("  Regla semántica: E.val = E₁.val + T.val  (sin orden explícito)")
print("\nComo Esquema de traducción (imperativo):")
print("  E → E₁ + T { E.val = E₁.val + T.val }")
print("  La acción está al final ↑ — se ejecuta al reducir")

**📝 Reflexión:** ¿En qué situación es preferible usar un esquema de traducción sobre una TDS declarativa, y viceversa?

---
## Ejercicio U1-09 (Práctico) — Generación de Código de Tres Direcciones

Una de las aplicaciones más importantes de la TDS es la **generación de código intermedio** en formato de tres direcciones (también llamado código TAC: *Three-Address Code*).

In [ ]:
# Ejercicio U1-09: Generación de código de tres direcciones (TAC)
# Cada instrucción TAC tiene la forma:  temp = operando1 op operando2

import re

class GeneradorTAC:
    def __init__(self, texto):
        self.tokens = re.findall(r'[a-zA-Z_]\w*|\d+\.?\d*|[+\-*/()]', texto.replace(' ', ''))
        self.pos = 0
        self.codigo = []
        self.contador_temp = 0

    def nuevo_temp(self):
        self.contador_temp += 1
        return f"t{self.contador_temp}"

    def emitir(self, instruccion):
        self.codigo.append(instruccion)

    def token_actual(self):
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None

    def consumir(self):
        tok = self.token_actual(); self.pos += 1; return tok

    def generar(self):
        resultado = self.E()
        return resultado

    def E(self):
        izq = self.T()
        while self.token_actual() in ('+', '-'):
            op = self.consumir()
            der = self.T()
            temp = self.nuevo_temp()
            self.emitir(f"{temp} = {izq} {op} {der}")  # ← TAC
            izq = temp
        return izq

    def T(self):
        izq = self.F()
        while self.token_actual() in ('*', '/'):
            op = self.consumir()
            der = self.F()
            temp = self.nuevo_temp()
            self.emitir(f"{temp} = {izq} {op} {der}")
            izq = temp
        return izq

    def F(self):
        tok = self.token_actual()
        if tok == '(':
            self.consumir(); val = self.E(); self.consumir(); return val
        return self.consumir()

# Generar TAC para varias expresiones
expresiones = [
    "a + b * c",
    "(a + b) * (c - d)",
    "x * x + y * y",
]

for expr in expresiones:
    g = GeneradorTAC(expr)
    resultado = g.generar()
    print(f"Expresión: {expr}")
    print(f"Código de tres direcciones:")
    for instruccion in g.codigo:
        print(f"  {instruccion}")
    print(f"  Resultado final en: {resultado}")
    print()

**📝 Tarea:** Agrega soporte para la potencia `**`. Por ejemplo, `a ** b` debe generar `t1 = a ** b`. ¿La potencia debe tener mayor o menor precedencia que `*`?

---
## Ejercicio U1-10 (Práctico) — Construcción de Árbol Sintáctico Abstracto (AST)

El **Árbol Sintáctico Abstracto (AST)** es una aplicación directa de TDS. A diferencia del árbol de análisis, omite nodos intermedios innecesarios.

In [ ]:
# Ejercicio U1-10: Construcción y evaluación de AST
import re

# Nodos del AST
class NodoOp:
    def __init__(self, op, izq, der):
        self.op = op; self.izq = izq; self.der = der

class NodoNum:
    def __init__(self, val):
        self.val = val

class NodoVar:
    def __init__(self, nombre):
        self.nombre = nombre

def evaluar_ast(nodo, env={}):
    if isinstance(nodo, NodoNum):
        return nodo.val
    elif isinstance(nodo, NodoVar):
        return env.get(nodo.nombre, 0)
    elif isinstance(nodo, NodoOp):
        izq = evaluar_ast(nodo.izq, env)
        der = evaluar_ast(nodo.der, env)
        if nodo.op == '+': return izq + der
        if nodo.op == '-': return izq - der
        if nodo.op == '*': return izq * der
        if nodo.op == '/': return izq / der

def imprimir_ast(nodo, nivel=0):
    prefijo = '  ' * nivel
    if isinstance(nodo, NodoOp):
        print(f"{prefijo}({nodo.op})")
        imprimir_ast(nodo.izq, nivel + 1)
        imprimir_ast(nodo.der, nivel + 1)
    elif isinstance(nodo, NodoNum):
        print(f"{prefijo}{nodo.val}")
    elif isinstance(nodo, NodoVar):
        print(f"{prefijo}[{nodo.nombre}]")

# Construimos manualmente el AST para: (a + 3) * (b - 1)
ast = NodoOp('*',
    NodoOp('+', NodoVar('a'), NodoNum(3)),
    NodoOp('-', NodoVar('b'), NodoNum(1))
)

print("AST para: (a + 3) * (b - 1)")
print("=" * 30)
imprimir_ast(ast)

# Evaluar con diferentes entornos de variables
print("\nEvaluaciones:")
entornos = [
    {'a': 2, 'b': 5},
    {'a': 10, 'b': 3},
    {'a': 0, 'b': 1},
]
for env in entornos:
    resultado = evaluar_ast(ast, env)
    print(f"  {env} → {resultado}")

**📝 Reflexión:** ¿Qué ventajas tiene un AST sobre el árbol de análisis completo generado por el parser? ¿Por qué los compiladores prefieren trabajar con AST?

---
## Ejercicio U1-11 (Teórico + Práctico) — Evaluación en Analizador Ascendente (Pila)

Los analizadores ascendentes (LR) usan una **pila semántica** para evaluar atributos sintetizados en el momento de la reducción.

In [ ]:
# Ejercicio U1-11: Simulación de pila semántica (analizador ascendente)
# Simulamos el proceso de reducción con una pila de atributos

def simular_pila_semantica(tokens_con_vals):
    """
    Simula la evaluación S-atribuida usando pila semántica.
    tokens_con_vals: lista de (token, valor_léxico)
    Gramática: E → E+T | T,  T → T*F | F,  F → num
    """
    pila_sint = []   # pila sintáctica
    pila_sem  = []   # pila semántica (atributos)
    pos = 0
    paso = 0

    def mostrar_estado(accion):
        nonlocal paso
        paso += 1
        sint_str = str(pila_sint)[-40:]
        sem_str  = str(pila_sem)[-30:]
        print(f"  Paso {paso:2d} | Pila: {sint_str:40} | Vals: {sem_str:25} | {accion}")

    print(f"Simulando: {' '.join(str(t[0]) for t in tokens_con_vals)}")
    print("-" * 110)

    # Evaluador simplificado: calcula la expresión paso a paso
    # (Simulación conceptual del proceso shift-reduce)
    operandos = []
    operadores = []

    prec = {'+': 1, '-': 1, '*': 2, '/': 2}

    def aplicar_op():
        op  = operadores.pop()
        der = operandos.pop()
        izq = operandos.pop()
        if op == '+': res = izq + der
        elif op == '-': res = izq - der
        elif op == '*': res = izq * der
        elif op == '/': res = izq / der
        operandos.append(res)
        mostrar_estado(f"REDUCE: {izq} {op} {der} = {res}")

    for tok, val in tokens_con_vals:
        if isinstance(val, (int, float)):
            operandos.append(val)
            mostrar_estado(f"SHIFT num({val}) → F.val = {val}")
        elif tok in prec:
            while (operadores and operadores[-1] in prec and
                   prec[operadores[-1]] >= prec[tok]):
                aplicar_op()
            operadores.append(tok)
            mostrar_estado(f"SHIFT operador '{tok}'")

    while operadores:
        aplicar_op()

    print(f"\n  → Resultado final: {operandos[0]}")

# Expresión: 2 + 3 * 4
simular_pila_semantica([
    ('num', 2), ('+', None), ('num', 3), ('*', None), ('num', 4)
])

**📝 Reflexión:** ¿Por qué los atributos sintetizados son perfectos para la pila semántica? ¿Qué problema habría si se necesitaran atributos heredados aquí?

---
## Ejercicio U1-12 (Teórico) — Quiz: Identificación de Atributos

Ejercicio de autoevaluación sobre la clasificación de atributos.

In [ ]:
# Ejercicio U1-12: Quiz interactivo de clasificación de atributos

preguntas = [
    {
        'produccion': 'E → E₁ + T',
        'regla': 'E.val = E₁.val + T.val',
        'atributo': 'E.val',
        'tipo_correcto': 'sintetizado',
        'explicacion': 'E.val se calcula desde sus HIJOS E₁ y T → flujo ascendente.'
    },
    {
        'produccion': 'D → T L',
        'regla': 'L.tipo_h = T.tipo',
        'atributo': 'L.tipo_h',
        'tipo_correcto': 'heredado',
        'explicacion': 'L.tipo_h recibe el valor desde T (hermano izquierdo) → flujo descendente.'
    },
    {
        'produccion': 'T → int',
        'regla': 'T.tipo = "integer"',
        'atributo': 'T.tipo',
        'tipo_correcto': 'sintetizado',
        'explicacion': 'T.tipo se calcula del terminal "int" (hijo) → sintetizado.'
    },
    {
        'produccion': 'L → L₁ , id',
        'regla': 'L₁.tipo_h = L.tipo_h',
        'atributo': 'L₁.tipo_h',
        'tipo_correcto': 'heredado',
        'explicacion': 'L₁.tipo_h recibe el valor del padre L → flujo descendente.'
    },
    {
        'produccion': 'F → ( E )',
        'regla': 'F.val = E.val',
        'atributo': 'F.val',
        'tipo_correcto': 'sintetizado',
        'explicacion': 'F.val se obtiene del hijo E → flujo ascendente.'
    },
]

print("🧠 QUIZ: Clasificación de Atributos")
print("="*60)
aciertos = 0

respuestas_usuario = ['sintetizado', 'heredado', 'sintetizado', 'heredado', 'sintetizado']
# ← En Colab: puedes cambiar estas respuestas para probar

for i, (q, r) in enumerate(zip(preguntas, respuestas_usuario)):
    correcto = r.lower() == q['tipo_correcto']
    if correcto:
        aciertos += 1
    icono = '✅' if correcto else '❌'
    print(f"\n  Pregunta {i+1}: {q['produccion']}")
    print(f"    Regla: {q['regla']}")
    print(f"    Tipo de {q['atributo']}: {r}  {icono}")
    if not correcto:
        print(f"    Correcto: {q['tipo_correcto']}")
    print(f"    💡 {q['explicacion']}")

print(f"\n{'='*60}")
print(f"Resultado: {aciertos}/{len(preguntas)} correctas")

---
## Ejercicio U1-13 (Práctico) — Verificación de Tipos con TDS (Aplicación)

La verificación de tipos es una aplicación directa de TDS que asigna un atributo `tipo` a cada nodo del árbol.

In [ ]:
# Ejercicio U1-13: Verificador de tipos simple usando TDS S-atribuida

class VerificadorTiposTDS:
    """
    TDS para verificar tipos en expresiones aritméticas.
    Reglas semánticas:
      E → E + E:  E.tipo = widening(E₁.tipo, E₂.tipo)
      E → num:    E.tipo = 'int'
      E → num.f:  E.tipo = 'float'
      E → id:     E.tipo = tablaSimbolos[id]
    """

    # Jerarquía de tipos: int puede ser promovido a float
    JERARQUIA = {'int': 1, 'float': 2, 'error': 0}

    TABLA_COMPATIBILIDAD = {
        ('+', 'int',   'int'):   'int',
        ('+', 'float', 'float'): 'float',
        ('+', 'int',   'float'): 'float',   # coerción
        ('+', 'float', 'int'):   'float',   # coerción
        ('+', 'bool',  'bool'):  'error',   # no válido
        ('*', 'int',   'int'):   'int',
        ('*', 'float', 'float'): 'float',
        ('*', 'int',   'float'): 'float',
        ('*', 'float', 'int'):   'float',
    }

    def __init__(self, tabla_simbolos):
        self.tabla = tabla_simbolos
        self.errores = []

    def tipo_operacion(self, op, t1, t2, contexto):
        """Regla semántica para operaciones binarias."""
        tipo = self.TABLA_COMPATIBILIDAD.get((op, t1, t2), 'error')
        if tipo == 'error':
            self.errores.append(f"Error semántico: no se puede aplicar '{op}' a {t1} y {t2} en: {contexto}")
        return tipo

    def tipo_identificador(self, nombre):
        if nombre not in self.tabla:
            self.errores.append(f"Error: identificador '{nombre}' no declarado")
            return 'error'
        return self.tabla[nombre]

# Simulación: tipo de cada subexpresión
tabla_simbolos = {'x': 'int', 'y': 'float', 'activo': 'bool', 'z': 'int'}
ver = VerificadorTiposTDS(tabla_simbolos)

print("Verificación de tipos (TDS S-atribuida):")
print("="*60)

# Expresión: x + y  (int + float → float, coerción)
t_x = ver.tipo_identificador('x')                    # E → id
t_y = ver.tipo_identificador('y')                    # E → id
t1  = ver.tipo_operacion('+', t_x, t_y, 'x + y')    # E → E + E
print(f"  x + y    → tipo(x)={t_x}, tipo(y)={t_y} → resultado: {t1}")

# Expresión: z * 3  (int * int → int)
t_z   = ver.tipo_identificador('z')
t_3   = 'int'                                        # num literal
t2    = ver.tipo_operacion('*', t_z, t_3, 'z * 3')
print(f"  z * 3    → tipo(z)={t_z}, tipo(3)={t_3} → resultado: {t2}")

# Expresión: activo + x  (bool + int → error)
t_act = ver.tipo_identificador('activo')
t3    = ver.tipo_operacion('+', t_act, t_x, 'activo + x')
print(f"  activo+x → tipo(activo)={t_act}, tipo(x)={t_x} → resultado: {t3}")

print("\n🔴 Errores detectados:")
if ver.errores:
    for err in ver.errores:
        print(f"  {err}")
else:
    print("  Ninguno")

---
## Ejercicio U1-14 (Práctico) — Esquema de Traducción para Números en Distintas Bases

Un esquema de traducción puede convertir representaciones de números en distintas bases a decimal.

In [ ]:
# Ejercicio U1-14: TDS para conversión de bases numéricas
# Gramática para números binarios:
#   num → num₁ bit  { num.val = num₁.val * 2 + bit.val }
#   num → bit       { num.val = bit.val }
#   bit → 0         { bit.val = 0 }
#   bit → 1         { bit.val = 1 }

def convertir_tds(cadena, base):
    """Convierte un número en la base dada a decimal usando lógica TDS."""
    digitos_validos = set('0123456789ABCDEF'[:base])
    cadena = cadena.upper()

    if not all(c in digitos_validos for c in cadena):
        print(f"  ❌ '{cadena}' contiene dígitos inválidos para base {base}")
        return None

    # Simulación TDS: num.val se calcula de izquierda a derecha
    # num → num₁ digito: val = val_anterior * base + digito_val
    val = 0  # atributo sintetizado acumulado
    pasos = []

    for digito in cadena:
        d_val = int(digito, 16)
        nuevo_val = val * base + d_val
        pasos.append(f"val={val}*{base}+{d_val} = {nuevo_val}")
        val = nuevo_val

    return val, pasos

print("TDS: Conversión de bases numéricas")
print("="*55)
casos = [
    ('1101',   2, 'Binario'),
    ('775',    8, 'Octal'),
    ('FF',    16, 'Hexadecimal'),
    ('1010',   2, 'Binario'),
    ('1A3',   16, 'Hexadecimal'),
]
for num_str, base, nombre_base in casos:
    resultado, pasos = convertir_tds(num_str, base)
    print(f"\n  {nombre_base} {num_str} (base {base}) → {resultado} decimal")
    print(f"  Reglas semánticas aplicadas:")
    for p in pasos:
        print(f"    → {p}")

**📝 Tarea:** ¿Cómo modificarías esta TDS para manejar números con punto decimal? Por ejemplo, convertir `101.11` en binario a decimal.

---
## Ejercicio U1-15 (Práctico) — Evaluación de Definición Dirigida con Grafos

Implementamos la detección de ciclos en grafos de dependencias.

In [ ]:
# Ejercicio U1-15: Detector de ciclos en grafo de dependencias

from collections import defaultdict

class GrafoDepCompleto:
    def __init__(self, nombre):
        self.nombre = nombre
        self.ady = defaultdict(list)
        self.nodos = set()

    def agregar_dep(self, desde, hacia):
        self.ady[desde].append(hacia)
        self.nodos.update([desde, hacia])

    def tiene_ciclo(self):
        """DFS para detectar ciclos."""
        color = {n: 'blanco' for n in self.nodos}
        ciclo_hallado = []

        def dfs(nodo, camino):
            color[nodo] = 'gris'
            for vecino in self.ady.get(nodo, []):
                if color.get(vecino) == 'gris':
                    idx = camino.index(vecino)
                    ciclo_hallado.append(camino[idx:] + [vecino])
                    return True
                if color.get(vecino, 'blanco') == 'blanco':
                    if dfs(vecino, camino + [vecino]):
                        return True
            color[nodo] = 'negro'
            return False

        for nodo in self.nodos:
            if color.get(nodo) == 'blanco':
                if dfs(nodo, [nodo]):
                    return True, ciclo_hallado[0]
        return False, []

    def analizar(self):
        print(f"\nAnálisis grafo: '{self.nombre}'")
        print(f"  Dependencias:")
        for n, vecinos in self.ady.items():
            for v in vecinos:
                print(f"    {n} ──▶ {v}")
        tiene, ciclo = self.tiene_ciclo()
        if tiene:
            print(f"  ❌ CICLO DETECTADO: {' → '.join(ciclo)}")
            print(f"  → Esta TDS NO es evaluable")
        else:
            print(f"  ✅ Sin ciclos — TDS válida y evaluable")

# Grafo 1: TDS válida (sin ciclos)
g1 = GrafoDepCompleto("Calculadora S-atribuida")
g1.agregar_dep('num.lexval', 'F.val')
g1.agregar_dep('F.val', 'T.val')
g1.agregar_dep('T.val', 'E.val')
g1.analizar()

# Grafo 2: TDS con ciclo (inválida)
g2 = GrafoDepCompleto("TDS con ciclo (inválida)")
g2.agregar_dep('A.x', 'B.y')
g2.agregar_dep('B.y', 'C.z')
g2.agregar_dep('C.z', 'A.x')  # ← ciclo
g2.analizar()

---
## Ejercicio U1-16 (Práctico) — Analizador Descendente con Herencia de Atributos

Implementamos un analizador recursivo descendente que usa atributos heredados como argumentos de función.

In [ ]:
# Ejercicio U1-16: Análisis descendente con atributos heredados
# Gramática para declaraciones múltiples:
#   programa → declaracion*
#   declaracion → tipo lista_ids ';'
#   lista_ids → id (',' id)*

import re

class AnalizadorDeclaraciones:
    TIPOS = {'int', 'float', 'bool', 'char', 'string'}

    def __init__(self, codigo):
        # Tokenizar respetando palabras clave e identificadores
        self.tokens = re.findall(r'[a-zA-Z_]\w*|;|,', codigo.replace(' ', ''))
        self.pos = 0
        self.tabla = {}   # tabla de símbolos resultante
        self.trazas = []

    def tok(self):
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None

    def consumir(self, esperado=None):
        t = self.tok()
        if esperado and t != esperado:
            raise SyntaxError(f"Esperado '{esperado}', encontrado '{t}'")
        self.pos += 1
        return t

    def programa(self):
        while self.tok() and self.tok() in self.TIPOS:
            self.declaracion()

    def declaracion(self):
        # Atributo sintetizado: tipo
        tipo_val = self.consumir()   # T.tipo (sintetizado del token)
        # tipo_val se HEREDA hacia lista_ids
        self.trazas.append(f"Tipo sintetizado: '{tipo_val}'")
        self.lista_ids(tipo_val)     # ← tipo heredado como argumento
        self.consumir(';')

    def lista_ids(self, tipo_heredado):
        """Recibe tipo_heredado — atributo heredado del padre."""
        nombre = self.consumir()
        self.tabla[nombre] = tipo_heredado
        self.trazas.append(f"  Insertar: {nombre} → {tipo_heredado} (heredado)")
        while self.tok() == ',':
            self.consumir(',')
            nombre = self.consumir()
            self.tabla[nombre] = tipo_heredado  # mismo tipo heredado
            self.trazas.append(f"  Insertar: {nombre} → {tipo_heredado} (heredado)")

# Código de ejemplo
codigo = "int x, y, z; float precio, iva; bool activo;"
analizador = AnalizadorDeclaraciones(codigo)
analizador.programa()

print(f"Código analizado: {codigo}")
print("\nTrazas de la TDS L-atribuida:")
for t in analizador.trazas:
    print(f"  {t}")

print("\nTabla de Símbolos resultante:")
print(f"  {'Identificador':15} | Tipo")
print(f"  {'-'*15}-+-{'-'*10}")
for nombre, tipo in analizador.tabla.items():
    print(f"  {nombre:15} | {tipo}")

---
## Ejercicio U1-17 (Teórico) — Mapa Conceptual de la Unidad 1

In [ ]:
# Ejercicio U1-17: Resumen visual de conceptos Unidad 1

conceptos = {
    'TDS (Traducción Dirigida por la Sintaxis)': {
        'def': 'Asocia semántica a la gramática mediante atributos y reglas semánticas',
        'componentes': ['Gramática libre del contexto', 'Atributos', 'Reglas semánticas'],
        'herramientas': ['YACC', 'Bison']
    },
    'Atributos': {
        'Sintetizados': 'Calculados desde los hijos (↑). Usados en S-atribuidas.',
        'Heredados':    'Calculados desde el padre y hermanos izquierdos (↓). Usados en L-atribuidas.'
    },
    'Tipos de TDS': {
        'S-atribuida': 'Solo sintetizados. Compatible con analizadores ascendentes (LR).',
        'L-atribuida': 'Sintetizados + heredados (solo izquierda). Compatible con descendentes (LL).'
    },
    'Herramientas de evaluación': {
        'Grafo de dependencias': 'Visualiza dependencias entre atributos. Detecta ciclos.',
        'Ordenación topológica': 'Determina un orden de evaluación válido.',
        'Pila semántica':        'Implementa S-atribuidas en analizadores ascendentes.'
    },
    'Esquema de traducción': {
        'def': 'TDS con acciones semánticas incrustadas en las producciones',
        'diferencia': 'Especifica CUÁNDO se ejecuta cada acción (orden explícito)',
        'posición': 'Al final → S-atribuida. En medio → L-atribuida'
    },
    'Aplicaciones': ['Construcción de AST', 'Verificación de tipos', 'Generación de código TAC', 'Conversión de bases']
}

print("📊 MAPA CONCEPTUAL — UNIDAD 1: TDS")
print("=" * 65)
for concepto, info in conceptos.items():
    print(f"\n🔷 {concepto}")
    if isinstance(info, dict):
        for k, v in info.items():
            if isinstance(v, list):
                print(f"   {k}: {', '.join(v)}")
            else:
                print(f"   {k}: {v}")
    elif isinstance(info, list):
        for item in info:
            print(f"   • {item}")

---
## Ejercicio U1-18 (Práctico) — Evaluador de Expresiones Booleanas con TDS

In [ ]:
# Ejercicio U1-18: TDS para expresiones booleanas
# Gramática:
#   B → B or B | B and B | not B | ( B ) | true | false | id
# Con precedencia: not > and > or

import re

class EvaluadorBooleano:
    def __init__(self, texto, entorno={}):
        self.tokens = re.findall(r'\bor\b|\band\b|\bnot\b|\btrue\b|\bfalse\b|[a-zA-Z_]\w*|[()]', texto)
        self.pos = 0
        self.env = entorno

    def tok(self): return self.tokens[self.pos] if self.pos < len(self.tokens) else None
    def consumir(self): t = self.tok(); self.pos += 1; return t

    def evaluar(self):
        val = self.B_or()
        return val

    def B_or(self):
        # B → B₁ or B₂ : B.val = B₁.val OR B₂.val (sintetizado)
        val = self.B_and()
        while self.tok() == 'or':
            self.consumir()
            val = val or self.B_and()  # regla semántica
        return val

    def B_and(self):
        # B → B₁ and B₂ : B.val = B₁.val AND B₂.val
        val = self.B_not()
        while self.tok() == 'and':
            self.consumir()
            val = val and self.B_not()
        return val

    def B_not(self):
        # B → not B₁ : B.val = NOT B₁.val
        if self.tok() == 'not':
            self.consumir()
            return not self.B_not()
        return self.B_base()

    def B_base(self):
        tok = self.tok()
        if tok == '(':
            self.consumir(); val = self.B_or(); self.consumir(); return val
        elif tok == 'true':  self.consumir(); return True
        elif tok == 'false': self.consumir(); return False
        else:
            self.consumir()
            return self.env.get(tok, False)

# Pruebas
entorno = {'p': True, 'q': False, 'r': True}
print(f"Entorno: {entorno}")
print("\nEvaluaciones booleanas:")
expresiones = [
    "p and q",
    "p or q",
    "not p",
    "not q",
    "p and not q",
    "(p or q) and r",
    "not (p and q)",
    "p or q and r",    # and tiene mayor precedencia que or
]
for expr in expresiones:
    ev = EvaluadorBooleano(expr, entorno)
    val = ev.evaluar()
    print(f"  {expr:30} → {val}")

---
## Ejercicio U1-19 (Práctico) — TDS para Análisis de Arrays

In [ ]:
# Ejercicio U1-19: TDS para calcular el tipo y tamaño de arrays
# Gramática para tipos:
#   T → array [ num ] of T  { T.tipo = array(num.val, T₁.tipo); T.ancho = num.val * T₁.ancho }
#   T → int                 { T.tipo = 'int'; T.ancho = 4 }
#   T → float               { T.tipo = 'float'; T.ancho = 8 }

class TipoBase:
    def __init__(self, nombre, ancho):
        self.nombre = nombre
        self.ancho = ancho
    def __str__(self):
        return f"{self.nombre}(ancho={self.ancho})"

class TipoArray:
    def __init__(self, tamaño, tipo_elem):
        self.tamaño = tamaño
        self.tipo_elem = tipo_elem
        self.ancho = tamaño * tipo_elem.ancho   # regla semántica: T.ancho = n * T₁.ancho

    def __str__(self):
        return f"array({self.tamaño}, {self.tipo_elem}) [ancho_total={self.ancho} bytes]"

# Aplicamos las reglas semánticas manualmente
tipos_base = {
    'int':   TipoBase('int', 4),
    'float': TipoBase('float', 8),
    'char':  TipoBase('char', 1),
    'bool':  TipoBase('bool', 1),
}

def construir_tipo(descripcion):
    """
    Construye la representación interna del tipo aplicando TDS.
    descripcion: lista como ['array', 10, 'of', 'int']
                          o ['array', 5, 'of', 'array', 3, 'of', 'float']
    """
    import re
    tokens = re.findall(r'array|of|\d+|int|float|char|bool', descripcion)
    pos = [0]

    def T():
        if tokens[pos[0]] == 'array':
            pos[0] += 1  # consumir 'array'
            n = int(tokens[pos[0]]); pos[0] += 1   # consumir tamaño
            pos[0] += 1  # consumir 'of'
            tipo_elem = T()   # T₁ (recursivo)
            return TipoArray(n, tipo_elem)   # regla semántica
        else:
            nombre = tokens[pos[0]]; pos[0] += 1
            return tipos_base[nombre]

    return T()

casos = [
    "array 10 of int",
    "array 5 of float",
    "array 3 of array 4 of int",    # array 2D: 3 × 4 × 4 = 48 bytes
    "array 2 of array 3 of array 4 of char",  # array 3D
]

print("TDS para tipos array:")
print("="*65)
for caso in casos:
    tipo = construir_tipo(caso)
    print(f"\n  Declaración: {caso}")
    print(f"  Tipo construido: {tipo}")

---
## Ejercicio U1-20 (Integrador) — Mini-compilador: Del código fuente al código TAC

Ejercicio integrador que combina: análisis léxico, análisis sintáctico con TDS y generación de código TAC.

In [ ]:
# Ejercicio U1-20: Mini-compilador integrador (análisis + TDS + TAC)

import re

class MiniCompilador:
    """Compila expresiones de asignación a código TAC con verificación básica."""

    def __init__(self):
        self.tabla = {}      # tabla de símbolos
        self.tac   = []      # código generado
        self.temps = 0
        self.errores = []

    def nuevo_temp(self):
        self.temps += 1
        return f"t{self.temps}"

    def tokenizar(self, linea):
        return re.findall(r'int|float|[a-zA-Z_]\w*|\d+\.?\d*|[=+\-*/;,]', linea)

    def compilar_programa(self, codigo):
        lineas = [l.strip() for l in codigo.strip().split('\n') if l.strip()]
        print("Compilando...\n")
        for linea in lineas:
            self.compilar_linea(linea)

    def compilar_linea(self, linea):
        tokens = self.tokenizar(linea)
        if not tokens: return

        # Declaración: int/float id, id;
        if tokens[0] in ('int', 'float'):
            tipo = tokens[0]
            for tok in tokens[1:]:
                if tok not in (',', ';'):
                    self.tabla[tok] = {'tipo': tipo, 'inicializado': False}
                    self.tac.append(f"; decl {tipo} {tok}")
        # Asignación: id = expr;
        elif len(tokens) >= 3 and tokens[1] == '=':
            var_dest = tokens[0]
            if var_dest not in self.tabla:
                self.errores.append(f"Error: '{var_dest}' no declarada")
                return
            expr_tokens = tokens[2:]
            if expr_tokens[-1] == ';': expr_tokens = expr_tokens[:-1]
            resultado = self.evaluar_expr(expr_tokens)
            if resultado:
                self.tac.append(f"{var_dest} = {resultado}")
                self.tabla[var_dest]['inicializado'] = True

    def evaluar_expr(self, tokens):
        """Genera TAC para una expresión simple (sin paréntesis)."""
        if len(tokens) == 1:
            return tokens[0]
        # Procesar operaciones respetando precedencia
        # Primero * y /, luego + y -
        for ops in [('*', '/'), ('+', '-')]:
            i = 1
            while i < len(tokens):
                if tokens[i] in ops:
                    izq = tokens[i-1]
                    op  = tokens[i]
                    der = tokens[i+1]
                    # Verificar variables
                    for v in [izq, der]:
                        if re.match(r'^[a-zA-Z_]', v) and v not in self.tabla:
                            self.errores.append(f"Error: '{v}' no declarada")
                    temp = self.nuevo_temp()
                    self.tac.append(f"{temp} = {izq} {op} {der}")
                    tokens = tokens[:i-1] + [temp] + tokens[i+2:]
                else:
                    i += 1
        return tokens[0]

# Programa de prueba
programa = """
int a, b, c;
float precio, total;
a = 5;
b = 3;
c = a + b * 2;
precio = 100;
total = precio * b + a;
"""

comp = MiniCompilador()
comp.compilar_programa(programa)

print("=" * 50)
print("CÓDIGO TAC GENERADO:")
print("=" * 50)
for instruccion in comp.tac:
    print(f"  {instruccion}")

print("\nTABLA DE SÍMBOLOS:")
for nombre, info in comp.tabla.items():
    estado = '✔' if info['inicializado'] else '○'
    print(f"  {estado} {nombre:10} | {info['tipo']}")

if comp.errores:
    print("\n🔴 ERRORES:")
    for e in comp.errores:
        print(f"  {e}")

---
---
# 🟢 UNIDAD 2: Declaración y Comprobación de Tipos
---

## Ejercicio U2-01 (Teórico) — Tabla de Símbolos: Estructura y Propósito

La **tabla de símbolos** es la estructura de datos central del compilador durante el análisis semántico. Almacena información sobre cada identificador declarado.

In [ ]:
# Ejercicio U2-01: Tabla de símbolos con hash y toda la información

class EntradaSimbolos:
    def __init__(self, nombre, tipo, ambito=0, direccion=None, info_extra=None):
        self.nombre    = nombre
        self.tipo      = tipo
        self.ambito    = ambito
        self.direccion = direccion
        self.info_extra = info_extra or {}

    def __repr__(self):
        return (f"EntradaSimbolos(nombre='{self.nombre}', tipo='{self.tipo}', "
                f"ambito={self.ambito}, dir={self.direccion})")

class TablaSimbolosHash:
    def __init__(self):
        self.tabla = {}   # implementación con dict de Python (tabla hash)
        self.contador_dir = 0
        self.tamaños_tipo = {'int': 4, 'float': 8, 'bool': 1, 'char': 1}

    def insertar(self, nombre, tipo, ambito=0, info_extra=None):
        if nombre in self.tabla:
            raise ValueError(f"'{nombre}' ya declarado en este ámbito")
        ancho = self.tamaños_tipo.get(tipo, 4)
        entrada = EntradaSimbolos(nombre, tipo, ambito, self.contador_dir, info_extra)
        self.tabla[nombre] = entrada
        self.contador_dir += ancho
        return entrada

    def buscar(self, nombre):
        return self.tabla.get(nombre)

    def mostrar(self):
        print(f"\n{'Nombre':15} | {'Tipo':8} | {'Ámbito':6} | {'Direc':6} | Info extra")
        print("-" * 65)
        for e in self.tabla.values():
            extra = str(e.info_extra) if e.info_extra else '—'
            print(f"  {e.nombre:13} | {e.tipo:8} | {e.ambito:6} | {e.direccion:6} | {extra}")

# Simular el procesamiento de declaraciones
t = TablaSimbolosHash()

# Variables simples
t.insertar('x', 'int')
t.insertar('y', 'float')
t.insertar('activo', 'bool')
t.insertar('letra', 'char')

# Función (con info extra: parámetros y tipo de retorno)
t.insertar('suma', 'int', info_extra={'params': ['int', 'int'], 'retorno': 'int'})
t.insertar('pi', 'float', info_extra={'constante': True, 'valor': 3.14159})

print("Tabla de Símbolos (implementación hash):")
t.mostrar()

# Operación de búsqueda
print("\nBúsqueda de 'y':")
entrada = t.buscar('y')
print(f"  {entrada}")

print("\nBúsqueda de 'z' (no declarada):")
print(f"  {t.buscar('z')}")

---
## Ejercicio U2-02 (Práctico) — Tabla de Símbolos con Ámbitos Anidados

Para gestionar bloques anidados, usamos una **pila de tablas hash**.

In [ ]:
# Ejercicio U2-02: Pila de tablas de símbolos para ámbitos anidados

class TablaSimbolosAmbitos:
    """Implementa la gestión de ámbitos con pila de tablas hash."""

    def __init__(self):
        self.pila = [{}]   # pila de diccionarios; tabla global en la base
        self.nivel = 0
        self.log = []

    def abrir_ambito(self):
        self.nivel += 1
        self.pila.append({})
        self.log.append(f"[Nivel {self.nivel}] abrir_ambito()")

    def cerrar_ambito(self):
        tabla_actual = self.pila.pop()
        self.log.append(f"[Nivel {self.nivel}] cerrar_ambito() → descartando: {list(tabla_actual.keys())}")
        self.nivel -= 1

    def insertar(self, nombre, tipo):
        if nombre in self.pila[-1]:
            self.log.append(f"  ⚠ '{nombre}' ya declarado en este ámbito (nivel {self.nivel})")
            return
        self.pila[-1][nombre] = tipo
        self.log.append(f"  ✔ insertar('{nombre}', '{tipo}') → nivel {self.nivel}")

    def buscar(self, nombre):
        # Búsqueda desde el ámbito más interno hacia el más externo
        for tabla in reversed(self.pila):
            if nombre in tabla:
                return tabla[nombre]
        return None  # no encontrado

    def mostrar_pila(self):
        print(f"Estado actual de la pila de ámbitos (nivel={self.nivel}):")
        for i, tabla in enumerate(self.pila):
            etiqueta = 'global' if i == 0 else f'bloque_{i}'
            print(f"  [{etiqueta}]: {tabla}")

# Simulamos el análisis de un programa con ámbitos anidados
ts = TablaSimbolosAmbitos()

# Ámbito global
ts.insertar('x', 'int')        # int x;
ts.insertar('y', 'float')      # float y;
print(">> Declaraciones globales")
ts.mostrar_pila()

# Entramos en una función
ts.abrir_ambito()
ts.insertar('x', 'bool')       # bool x; — ¡mismo nombre! → nueva variable local
ts.insertar('local', 'int')    # int local;
print("\n>> Dentro de función")
ts.mostrar_pila()

# Bloque anidado (if/for)
ts.abrir_ambito()
ts.insertar('i', 'int')        # int i;
print("\n>> Dentro de bloque anidado")
ts.mostrar_pila()

# Búsquedas
print("\n>> Búsquedas (desde el ámbito más interno):")
for nombre in ['i', 'local', 'x', 'y', 'z']:
    tipo = ts.buscar(nombre)
    print(f"  buscar('{nombre}') → {tipo if tipo else '❌ No encontrado'}")

# Cerramos ámbitos
ts.cerrar_ambito()   # salimos del bloque anidado
ts.cerrar_ambito()   # salimos de la función

print("\n>> Log completo:")
for linea in ts.log:
    print(f"  {linea}")

---
## Ejercicio U2-03 (Práctico) — Expresiones de Tipo

Los compiladores representan los tipos internamente como **expresiones de tipo**: árboles que describen la estructura del tipo.

In [ ]:
# Ejercicio U2-03: Representación de expresiones de tipo

class ExpresionTipo:
    """Clase base para expresiones de tipo."""
    def ancho(self): return 0
    def __eq__(self, otro): return str(self) == str(otro)

class TipoBasico(ExpresionTipo):
    ANCHOS = {'integer': 4, 'float': 8, 'char': 1, 'boolean': 1, 'error': 0}
    def __init__(self, nombre):
        self.nombre = nombre
    def ancho(self): return self.ANCHOS.get(self.nombre, 4)
    def __str__(self): return self.nombre

class TipoArray(ExpresionTipo):
    def __init__(self, tamaño, tipo_elem):
        self.tamaño = tamaño
        self.tipo_elem = tipo_elem
    def ancho(self): return self.tamaño * self.tipo_elem.ancho()
    def __str__(self): return f"array({self.tamaño}, {self.tipo_elem})"

class TipoPuntero(ExpresionTipo):
    def __init__(self, tipo_base):
        self.tipo_base = tipo_base
    def ancho(self): return 8  # puntero siempre 8 bytes (64 bits)
    def __str__(self): return f"pointer({self.tipo_base})"

class TipoFuncion(ExpresionTipo):
    def __init__(self, tipo_param, tipo_retorno):
        self.tipo_param = tipo_param
        self.tipo_retorno = tipo_retorno
    def ancho(self): return 0
    def __str__(self): return f"{self.tipo_param} → {self.tipo_retorno}"

class TipoProducto(ExpresionTipo):
    def __init__(self, tipos):
        self.tipos = tipos
    def ancho(self): return sum(t.ancho() for t in self.tipos)
    def __str__(self): return " × ".join(str(t) for t in self.tipos)

# Construimos expresiones de tipo
int_t   = TipoBasico('integer')
float_t = TipoBasico('float')
char_t  = TipoBasico('char')

tipos = [
    ('int',                    int_t),
    ('array(10, int)',         TipoArray(10, int_t)),
    ('array(3, array(4, float))', TipoArray(3, TipoArray(4, float_t))),
    ('pointer(int)',           TipoPuntero(int_t)),
    ('int → float',            TipoFuncion(int_t, float_t)),
    ('int × float',            TipoProducto([int_t, float_t])),
    ('array(5, pointer(int))', TipoArray(5, TipoPuntero(int_t))),
]

print(f"{'Expresión de tipo':40} | {'Ancho (bytes)':15}")
print("-" * 60)
for desc, tipo in tipos:
    print(f"  {str(tipo):38} | {tipo.ancho()}")

---
## Ejercicio U2-04 (Práctico) — Equivalencia Estructural vs Nominal

Dos tipos pueden ser **estructuralmente equivalentes** (misma estructura) pero **nominalmente distintos** (diferente nombre).

In [ ]:
# Ejercicio U2-04: Equivalencia estructural vs nominal de tipos

class SistemaEquivalencia:

    def __init__(self, modo='estructural'):
        self.modo = modo  # 'estructural' o 'nominal'
        self.tipos_nombrados = {}  # nombre → expresión de tipo

    def definir_tipo(self, nombre, expresion):
        """Equivale a: type nombre = expresion"""
        self.tipos_nombrados[nombre] = {'expr': expresion, 'id': id(expresion)}
        print(f"  Definido: type {nombre} = {expresion}")

    def son_equivalentes(self, nombre1, nombre2):
        t1 = self.tipos_nombrados.get(nombre1)
        t2 = self.tipos_nombrados.get(nombre2)
        if not t1 or not t2:
            return False, "tipo no definido"

        if self.modo == 'nominal':
            # Equivalencia nominal: solo si comparten el MISMO objeto
            eq = t1['id'] == t2['id']
            razon = "mismo objeto en memoria" if eq else "objetos distintos (aunque misma estructura)"
        else:
            # Equivalencia estructural: compara la representación textual
            eq = t1['expr'] == t2['expr']
            razon = f"'{t1['expr']}' == '{t2['expr']}'" if eq else f"'{t1['expr']}' ≠ '{t2['expr']}'"

        return eq, razon

# Definir tipos
int_t   = TipoBasico('integer')
float_t = TipoBasico('float')

# Mismo contenido, distinta creación
arr10_int_a = TipoArray(10, int_t)
arr10_int_b = TipoArray(10, TipoBasico('integer'))  # mismo contenido
arr5_float  = TipoArray(5, float_t)

for modo in ['estructural', 'nominal']:
    sys = SistemaEquivalencia(modo)
    print(f"\n{'='*55}")
    print(f"Modo: {modo.upper()}")
    print(f"{'='*55}")

    sys.definir_tipo('T1', arr10_int_a)   # type T1 = array(10, int)
    sys.definir_tipo('T2', arr10_int_b)   # type T2 = array(10, int) — mismo contenido
    sys.definir_tipo('T3', arr5_float)    # type T3 = array(5, float)
    sys.definir_tipo('T4', arr10_int_a)   # type T4 = T1 (mismo objeto)

    print()
    pares = [('T1', 'T2'), ('T1', 'T3'), ('T1', 'T4')]
    for a, b in pares:
        eq, razon = sys.son_equivalentes(a, b)
        icono = '✅' if eq else '❌'
        print(f"  {icono} {a} ≡ {b}? {eq:5} | {razon}")

---
## Ejercicio U2-05 (Práctico) — Comprobador de Tipos Completo

In [ ]:
# Ejercicio U2-05: Comprobador de tipos con coerciones

class ComprobadorTipos:
    """Verifica tipos en expresiones con coerciones implícitas."""

    # Reglas de coerción: (tipo_origen, tipo_destino) → es_válido
    COERCIONES = {
        ('int', 'float'): True,   # int se puede promover a float
        ('int', 'int'):   True,
        ('float', 'float'): True,
        ('float', 'int'): False,  # float NO puede bajar a int sin cast explícito
        ('char', 'int'):  True,   # char → int (valor ASCII)
        ('bool', 'int'):  False,  # bool → int no permitido (lenguaje estricto)
    }

    # Tipo resultante de operaciones binarias
    RESULTADO_OP = {
        ('int',   '+', 'int'):   'int',
        ('float', '+', 'float'): 'float',
        ('int',   '+', 'float'): 'float',
        ('float', '+', 'int'):   'float',
        ('int',   '*', 'int'):   'int',
        ('float', '*', 'float'): 'float',
        ('int',   '*', 'float'): 'float',
        ('float', '*', 'int'):   'float',
        ('int',   '<', 'int'):   'bool',
        ('float', '<', 'float'): 'bool',
        ('int',   '<', 'float'): 'bool',
        ('float', '<', 'int'):   'bool',
        ('int',   '==', 'int'):  'bool',
        ('float', '==', 'float'):'bool',
    }

    def __init__(self, tabla_simbolos):
        self.tabla = tabla_simbolos
        self.errores = []
        self.advertencias = []

    def tipo_expr(self, expr):
        """Simula E.tipo para una expresión (op: (tipo1, op, tipo2) o solo tipo)."""
        if isinstance(expr, str):
            return self.tabla.get(expr, {}).get('tipo', 'error')
        if isinstance(expr, tuple) and len(expr) == 3:
            t1, op, t2 = expr
            t1 = self.tipo_expr(t1)
            t2 = self.tipo_expr(t2)
            tipo_res = self.RESULTADO_OP.get((t1, op, t2))
            if tipo_res is None:
                self.errores.append(f"Error: operación '{op}' no válida entre {t1} y {t2}")
                return 'error'
            if t1 != t2:
                self.advertencias.append(f"Coerción implícita: {t1} → promovido a {t2} en '{op}'")
            return tipo_res
        return 'error'

    def verificar_asignacion(self, var_dest, expr):
        tipo_dest = self.tabla.get(var_dest, {}).get('tipo', 'error')
        tipo_expr_val = self.tipo_expr(expr)
        es_valido = self.COERCIONES.get((tipo_expr_val, tipo_dest), False)
        if not es_valido:
            self.errores.append(f"Error: no se puede asignar {tipo_expr_val} a {var_dest} ({tipo_dest})")
        return es_valido, tipo_expr_val, tipo_dest

tabla_s = {
    'a': {'tipo': 'int'}, 'b': {'tipo': 'int'},
    'x': {'tipo': 'float'}, 'flag': {'tipo': 'bool'},
    'resultado': {'tipo': 'float'}, 'n': {'tipo': 'int'},
}
comp = ComprobadorTipos(tabla_s)

print("Comprobación de tipos:")
print("=" * 60)

casos = [
    # (descripción, expresión_arbol, variable_destino)
    ("resultado = a + x",   ('a', '+', 'x'),           'resultado'),
    ("n = a + b",           ('a', '+', 'b'),            'n'),
    ("resultado = a * b",   ('a', '*', 'b'),            'resultado'),   # int → float: OK
    ("n = a + x",           ('a', '+', 'x'),            'n'),           # float → int: error
    ("flag = a + b",        ('a', '+', 'b'),            'flag'),        # int → bool: error
    ("n = a < b",           ('a', '<', 'b'),            'n'),           # bool → int: error
]

for desc, expr, dest in casos:
    ok, t_e, t_d = comp.verificar_asignacion(dest, expr)
    icono = '✅' if ok else '❌'
    print(f"  {icono} {desc:30} | expr_tipo={t_e:7} | dest_tipo={t_d}")

print("\n🔴 Errores:")
for e in comp.errores: print(f"  {e}")
print("\n⚠ Advertencias:")
for w in comp.advertencias: print(f"  {w}")

---
## Ejercicio U2-06 (Práctico) — Verificador de Tipos para Sentencias

In [ ]:
# Ejercicio U2-06: Verificador semántico de sentencias
# Verifica: asignaciones, if-then, while, funciones

class VerificadorSentencias:
    """
    Implementa las reglas semánticas de la unidad:
      S → id := E    { verificar tipo(id) compatible con E.tipo }
      S → if E then S { verificar E.tipo = boolean }
      S → while E do S { verificar E.tipo = boolean }
    """

    def __init__(self):
        self.tabla = {}
        self.errores = []
        self.paso = 0

    def declarar(self, nombre, tipo):
        self.tabla[nombre] = tipo

    def tipo_expr(self, expr):
        """Calcula E.tipo para una expresión."""
        if isinstance(expr, bool): return 'boolean'
        if isinstance(expr, int):  return 'integer'
        if isinstance(expr, float):return 'float'
        if isinstance(expr, str):
            if expr not in self.tabla:
                self.errores.append(f"'{expr}' no declarada")
                return 'error'
            return self.tabla[expr]
        if isinstance(expr, dict):
            op = expr.get('op')
            t1 = self.tipo_expr(expr['izq'])
            t2 = self.tipo_expr(expr['der'])
            if op in ('+', '-', '*', '/'):
                if t1 in ('integer', 'float') and t2 in ('integer', 'float'):
                    return 'float' if 'float' in (t1, t2) else 'integer'
                self.errores.append(f"Tipos no numéricos en '{op}': {t1}, {t2}")
                return 'error'
            if op in ('<', '>', '==', '!=', '<=', '>='):
                if t1 in ('integer', 'float') and t2 in ('integer', 'float'):
                    return 'boolean'
                self.errores.append(f"Comparación inválida: {t1} {op} {t2}")
                return 'error'
        return 'error'

    def verificar_asignacion(self, var, expr, linea=''):
        tipo_var  = self.tabla.get(var, 'error')
        tipo_expr = self.tipo_expr(expr)
        compat = (tipo_var == tipo_expr or
                  (tipo_var == 'float' and tipo_expr == 'integer'))
        self._reportar(f"{var} := {expr}", compat,
                       f"tipo({var})={tipo_var}, tipo(expr)={tipo_expr}")
        if not compat:
            self.errores.append(f"Asignación inválida: {tipo_expr} → {tipo_var} para '{var}'")

    def verificar_if(self, cond_expr, linea=''):
        tipo_cond = self.tipo_expr(cond_expr)
        ok = tipo_cond == 'boolean'
        self._reportar(f"if {cond_expr}", ok,
                       f"tipo_cond={tipo_cond} (requerido: boolean)")
        if not ok:
            self.errores.append(f"Condición 'if' debe ser boolean, es {tipo_cond}")

    def verificar_while(self, cond_expr):
        tipo_cond = self.tipo_expr(cond_expr)
        ok = tipo_cond == 'boolean'
        self._reportar(f"while {cond_expr}", ok,
                       f"tipo_cond={tipo_cond} (requerido: boolean)")
        if not ok:
            self.errores.append(f"Condición 'while' debe ser boolean, es {tipo_cond}")

    def _reportar(self, stmt, ok, detalle):
        self.paso += 1
        icono = '✅' if ok else '❌'
        print(f"  {icono} [{self.paso}] {stmt:35} | {detalle}")

# Uso del verificador
ver = VerificadorSentencias()
ver.declarar('x', 'integer'); ver.declarar('y', 'float')
ver.declarar('flag', 'boolean'); ver.declarar('n', 'integer')

print("Verificación de sentencias:")
print("=" * 75)

# Asignaciones
ver.verificar_asignacion('x', 5)                     # integer := integer ✓
ver.verificar_asignacion('y', 3)                     # float := integer ✓ (coerción)
ver.verificar_asignacion('x', 3.14)                  # integer := float ✗
ver.verificar_asignacion('flag', {'op': '<', 'izq': 'x', 'der': 'n'})   # bool := bool ✓
ver.verificar_asignacion('x', {'op': '+', 'izq': 'x', 'der': 'y'})     # int := float ✗

# Condicionales
ver.verificar_if('flag')                             # boolean ✓
ver.verificar_if({'op': '<', 'izq': 'x', 'der': 5}) # boolean ✓
ver.verificar_if('x')                               # integer ✗

# While
ver.verificar_while({'op': '!=', 'izq': 'n', 'der': 0})  # boolean ✓

print("\n🔴 Errores semánticos detectados:")
for e in ver.errores:
    print(f"  {e}")

---
## Ejercicio U2-07 (Práctico) — Resolución de Sobrecarga de Operadores

In [ ]:
# Ejercicio U2-07: Resolución de sobrecarga de operadores

class ResolutorSobrecarga:
    """
    Resuelve qué función/operador invocar cuando hay sobrecarga.
    El compilador debe:
    1. Encontrar candidatos cuyos tipos coincidan
    2. Si hay exactamente 1 → éxito
    3. Si hay 0 → error (no existe)
    4. Si hay >1 → error (ambiguo)
    """

    def __init__(self):
        self.operadores = {}  # nombre_op → lista de (tipos_param, tipo_resultado, descripción)

    def registrar_operador(self, nombre, tipos_param, tipo_resultado, descripcion):
        if nombre not in self.operadores:
            self.operadores[nombre] = []
        self.operadores[nombre].append((tipos_param, tipo_resultado, descripcion))

    def resolver(self, nombre_op, tipos_args):
        candidatos = self.operadores.get(nombre_op, [])
        # Buscar coincidencias exactas
        exactos = [(res, desc) for params, res, desc in candidatos if params == tipos_args]
        # Buscar con coerciones (int → float)
        def compatible(params, args):
            if len(params) != len(args): return False
            coerciones_ok = {('int', 'float'), ('int', 'int'), ('float', 'float')}
            return all((a, p) in coerciones_ok for a, p in zip(args, params))
        compat = [(res, desc) for params, res, desc in candidatos
                  if compatible(params, tipos_args) and params != tipos_args]

        print(f"\n  Resolver: {nombre_op}({', '.join(tipos_args)})")
        if exactos:
            res, desc = exactos[0]
            print(f"    ✅ Coincidencia exacta: {desc} → {res}")
            return res
        elif len(compat) == 1:
            res, desc = compat[0]
            print(f"    ✅ Resuelto con coerción: {desc} → {res}")
            return res
        elif len(compat) > 1:
            print(f"    ❌ AMBIGUO: {len(compat)} candidatos posibles")
            return 'error'
        else:
            print(f"    ❌ NO encontrado: sin versión de '{nombre_op}' para {tipos_args}")
            return 'error'

r = ResolutorSobrecarga()

# Registrar versiones del operador '+'
r.registrar_operador('+', ['int',   'int'],   'int',   '+ entera')
r.registrar_operador('+', ['float', 'float'], 'float', '+ flotante')
r.registrar_operador('+', ['str',   'str'],   'str',   '+ concatenación')

# Registrar función sobrecargada 'imprimir'
r.registrar_operador('imprimir', ['int'],   'void', 'imprimir(int)')
r.registrar_operador('imprimir', ['float'], 'void', 'imprimir(float)')
r.registrar_operador('imprimir', ['str'],   'void', 'imprimir(str)')

print("Resolución de sobrecarga:")
print("=" * 60)

# Pruebas
r.resolver('+', ['int', 'int'])       # exacto
r.resolver('+', ['int', 'float'])     # coerción
r.resolver('+', ['str', 'str'])       # exacto
r.resolver('+', ['bool', 'int'])      # no encontrado
r.resolver('imprimir', ['int'])       # exacto
r.resolver('imprimir', ['bool'])      # no encontrado

---
## Ejercicio U2-08 (Práctico) — Conversiones de Tipo Implícitas y Explícitas

In [ ]:
# Ejercicio U2-08: Gestión de coerciones en el AST

class GestorCoerciones:
    """Inserta nodos de conversión en el AST cuando se necesita coerción."""

    # Grafo de coerciones implícitas válidas: origen → destinos posibles
    COERCIONES_IMPLICITAS = {
        'int':   {'float', 'int'},
        'float': {'float'},
        'char':  {'int', 'char'},
        'bool':  {'bool'},
    }

    def tipo_resultado(self, t1, op, t2):
        """Determina el tipo del resultado de t1 op t2 con posibles coerciones."""
        pares_numericos = {
            ('int', 'int'): 'int',
            ('int', 'float'): 'float',
            ('float', 'int'): 'float',
            ('float', 'float'): 'float',
        }
        if op in ('+', '-', '*', '/'):
            tipo_res = pares_numericos.get((t1, t2))
            if tipo_res:
                coerciones = []
                if t1 != tipo_res: coerciones.append(f"coercionar({t1} → {tipo_res})")
                if t2 != tipo_res: coerciones.append(f"coercionar({t2} → {tipo_res})")
                return tipo_res, coerciones
            return 'error', [f"No hay coerción implícita entre {t1} y {t2}"]
        return 'error', []

    def verificar_casting_explicito(self, tipo_origen, tipo_destino):
        """Verifica si un cast explícito (tipo_destino)expr es válido."""
        # Castings explícitos válidos
        castings_validos = {
            ('float', 'int'), ('int', 'float'), ('char', 'int'), ('int', 'char'),
            ('int', 'bool'), ('bool', 'int'), ('float', 'char')
        }
        return (tipo_origen, tipo_destino) in castings_validos

gc = GestorCoerciones()
print("Análisis de coerciones implícitas:")
print("=" * 65)
ops = [
    ('int', '+', 'int'),
    ('int', '+', 'float'),
    ('float', '*', 'int'),
    ('bool', '+', 'int'),
    ('float', '-', 'float'),
]
for t1, op, t2 in ops:
    tipo_res, coerc = gc.tipo_resultado(t1, op, t2)
    coerc_str = ', '.join(coerc) if coerc else 'ninguna'
    icono = '✅' if tipo_res != 'error' else '❌'
    print(f"  {icono} {t1} {op} {t2} → {tipo_res:7} | coerciones: {coerc_str}")

print("\nCasting explícito (float) expr y (int) expr:")
castings = [
    ('float', 'int', "(int) 3.14"),
    ('int', 'float', "(float) 5"),
    ('bool', 'int', "(int) true"),
    ('int', 'bool', "(bool) 1"),
    ('char', 'float', "(float) 'a'"),
]
for orig, dest, desc in castings:
    ok = gc.verificar_casting_explicito(orig, dest)
    icono = '✅' if ok else '❌'
    print(f"  {icono} {desc:20} | ({orig} → {dest})")

---
## Ejercicio U2-09 (Práctico) — Análisis de Alcance (Scope Analysis)

In [ ]:
# Ejercicio U2-09: Análisis de alcance léxico (scope analysis)

class AnalizadorAlcance:
    """
    Analiza un programa simple e identifica:
    - Dónde se declara cada variable
    - Dónde es visible
    - Qué referencias son válidas e inválidas
    """

    def __init__(self):
        self.pila_ambitos = [{}]   # pila de ámbitos
        self.nivel = 0
        self.declaraciones = []    # rastro de declaraciones
        self.referencias    = []   # rastro de usos
        self.errores        = []

    def abrir(self, nombre='bloque'):
        self.nivel += 1
        self.pila_ambitos.append({})
        self.declaraciones.append({'tipo': 'abrir', 'nivel': self.nivel, 'nombre': nombre})

    def cerrar(self):
        self.declaraciones.append({'tipo': 'cerrar', 'nivel': self.nivel})
        self.pila_ambitos.pop()
        self.nivel -= 1

    def declarar(self, nombre, tipo):
        if nombre in self.pila_ambitos[-1]:
            self.errores.append(f"Error: '{nombre}' ya declarado en nivel {self.nivel}")
            return
        self.pila_ambitos[-1][nombre] = {'tipo': tipo, 'nivel': self.nivel}
        self.declaraciones.append({'tipo': 'decl', 'nombre': nombre, 'tipo_var': tipo, 'nivel': self.nivel})

    def usar(self, nombre, linea=0):
        for amb in reversed(self.pila_ambitos):
            if nombre in amb:
                info = amb[nombre]
                self.referencias.append({'nombre': nombre, 'nivel_uso': self.nivel,
                                          'nivel_decl': info['nivel'], 'ok': True, 'linea': linea})
                return info['tipo']
        self.errores.append(f"Error: '{nombre}' no declarada (línea {linea})")
        self.referencias.append({'nombre': nombre, 'nivel_uso': self.nivel, 'ok': False, 'linea': linea})
        return 'error'

# Simulamos el análisis de este programa:
# int x;
# { int y; x = y + 1; }
# { float z; z = x * 2.0; y = 1; }  ← y ya no es visible, error

a = AnalizadorAlcance()
a.declarar('x', 'int')         # declaración global

a.abrir('bloque_1')
a.declarar('y', 'int')         # local al bloque_1
a.usar('x', linea=3)           # ✓ x es global
a.usar('y', linea=3)           # ✓ y está en este ámbito
a.cerrar()

a.abrir('bloque_2')
a.declarar('z', 'float')       # local al bloque_2
a.usar('z', linea=5)           # ✓ z está en este ámbito
a.usar('x', linea=5)           # ✓ x es global
a.usar('y', linea=5)           # ✗ y ya no es visible!
a.cerrar()

# Mostrar trazas
print("Análisis de alcance:")
print("=" * 60)
for d in a.declaraciones:
    nivel = d.get('nivel', 0)
    ind = '  ' * nivel
    if d['tipo'] == 'abrir':
        print(f"{ind}▼ Entrar {d['nombre']} (nivel {nivel})")
    elif d['tipo'] == 'cerrar':
        print(f"{ind}▲ Salir (nivel {nivel})")
    else:
        print(f"{ind}  decl: {d['nombre']} : {d['tipo_var']}")

print("\nReferencias:")
for r in a.referencias:
    icono = '✅' if r['ok'] else '❌'
    nivel_decl = r.get('nivel_decl', '—')
    print(f"  {icono} usar('{r['nombre']}') línea={r['linea']} | nivel_uso={r['nivel_uso']}, declarado_en={nivel_decl}")

print("\n🔴 Errores:")
for e in a.errores: print(f"  {e}")

---
## Ejercicio U2-10 (Práctico) — Verificador de Tipos para Funciones

In [ ]:
# Ejercicio U2-10: Verificación de tipos en llamadas a funciones

class VerificadorFunciones:
    def __init__(self):
        self.funciones = {}   # nombre → {params: [tipos], retorno: tipo}
        self.errores = []

    def declarar_funcion(self, nombre, tipos_params, tipo_retorno):
        self.funciones[nombre] = {'params': tipos_params, 'retorno': tipo_retorno}
        sig = f"{nombre}({', '.join(tipos_params)}) → {tipo_retorno}"
        print(f"  Declarada: {sig}")

    def verificar_llamada(self, nombre_func, tipos_args):
        if nombre_func not in self.funciones:
            self.errores.append(f"'{nombre_func}' no declarada")
            return 'error'

        func = self.funciones[nombre_func]
        params = func['params']

        if len(tipos_args) != len(params):
            self.errores.append(f"'{nombre_func}': esperados {len(params)} args, recibidos {len(tipos_args)}")
            return 'error'

        # Verificar compatibilidad de tipos
        coerciones_ok = {('int', 'float'), ('int', 'int'), ('float', 'float')}
        errores_param = []
        for i, (arg, param) in enumerate(zip(tipos_args, params)):
            if arg != param and (arg, param) not in coerciones_ok:
                errores_param.append(f"arg{i+1}: esperado {param}, recibido {arg}")

        if errores_param:
            self.errores.append(f"'{nombre_func}': {'; '.join(errores_param)}")
            return 'error'

        return func['retorno']

    def verificar_retorno(self, nombre_func, tipo_expr_retorno):
        func = self.funciones.get(nombre_func)
        if not func:
            return False
        tipo_esperado = func['retorno']
        ok = (tipo_expr_retorno == tipo_esperado or
              (tipo_expr_retorno == 'int' and tipo_esperado == 'float'))
        if not ok:
            self.errores.append(f"Return de '{nombre_func}': esperado {tipo_esperado}, es {tipo_expr_retorno}")
        return ok

ver = VerificadorFunciones()

print("Declaración de funciones:")
ver.declarar_funcion('suma',    ['int', 'int'],   'int')
ver.declarar_funcion('promedio', ['float', 'float'], 'float')
ver.declarar_funcion('es_par',  ['int'],          'bool')
ver.declarar_funcion('max3',    ['int', 'int', 'int'], 'int')

print("\nVerificación de llamadas:")
print("-" * 70)
llamadas = [
    ('suma',     ['int', 'int'],     'suma(a, b)'),
    ('suma',     ['int', 'float'],   'suma(a, x)   ← coerción float→int'),
    ('promedio', ['int', 'float'],   'promedio(n, x) ← coerción int→float'),
    ('es_par',   ['int'],            'es_par(5)'),
    ('es_par',   ['float'],          'es_par(3.14)  ← error tipo'),
    ('max3',     ['int', 'int'],     'max3(a, b)    ← error aridad'),
    ('sqrt',     ['float'],          'sqrt(x)       ← no declarada'),
]
for nombre, args, desc in llamadas:
    tipo_ret = ver.verificar_llamada(nombre, args)
    icono = '✅' if tipo_ret != 'error' else '❌'
    print(f"  {icono} {desc:40} → retorna: {tipo_ret}")

print("\nVerificación de sentencias return:")
retornos = [
    ('suma', 'int', "return a+b (int) en suma"),
    ('suma', 'float', "return 3.14 en suma ← error"),
    ('promedio', 'float', "return prom (float) en promedio"),
    ('es_par', 'bool', "return flag (bool) en es_par"),
]
for func, tipo_ret, desc in retornos:
    ok = ver.verificar_retorno(func, tipo_ret)
    print(f"  {'✅' if ok else '❌'} {desc}")

print("\n🔴 Errores:")
for e in ver.errores: print(f"  {e}")

---
## Ejercicio U2-11 (Teórico) — Sistemas de Tipos: Propiedades

In [ ]:
# Ejercicio U2-11: Propiedades de sistemas de tipos

propiedades = {
    'Tipado estático': {
        'descripcion': 'Los tipos se verifican en tiempo de compilación',
        'ejemplos': ['C', 'Java', 'Rust', 'Haskell'],
        'ventajas': ['Errores detectados antes de ejecutar', 'Mejor rendimiento'],
        'desventajas': ['Menos flexible', 'Requiere más anotaciones'],
    },
    'Tipado dinámico': {
        'descripcion': 'Los tipos se verifican en tiempo de ejecución',
        'ejemplos': ['Python', 'JavaScript', 'Ruby', 'Lisp'],
        'ventajas': ['Mayor flexibilidad', 'Menos código'],
        'desventajas': ['Errores en tiempo de ejecución', 'Menor rendimiento'],
    },
    'Tipado fuerte': {
        'descripcion': 'No permite conversiones implícitas entre tipos incompatibles',
        'ejemplos': ['Python', 'Java', 'Haskell'],
        'ventajas': ['Mayor seguridad', 'Menos comportamiento inesperado'],
        'desventajas': ['Más verboso con conversiones explícitas'],
    },
    'Tipado débil': {
        'descripcion': 'Permite conversiones implícitas más permisivas',
        'ejemplos': ['C', 'JavaScript', 'PHP'],
        'ventajas': ['Mayor comodidad en ciertos casos'],
        'desventajas': ['Comportamientos sorprendentes: "1" + 1 = "11" en JS'],
    },
}

print("🏷 SISTEMAS DE TIPOS — Clasificación")
print("=" * 70)
for prop, info in propiedades.items():
    print(f"\n📌 {prop}")
    print(f"   Definición: {info['descripcion']}")
    print(f"   Lenguajes:  {', '.join(info['ejemplos'])}")
    print(f"   ✅ Ventajas: {'; '.join(info['ventajas'])}")
    print(f"   ⚠ Desventajas: {'; '.join(info['desventajas'])}")

print("\n" + "=" * 70)
print("Ejemplos de comportamientos por sistema de tipos:")
ejemplos_codigo = [
    ("Python (fuerte/dinámico)", '"3" + 3', 'TypeError en tiempo de ejecución'),
    ("JavaScript (débil/dinámico)", '"3" + 3', '"33" — coerción implícita a string'),
    ("Java (fuerte/estático)", 'int x = 3.14', 'Error en compilación'),
    ("C (débil/estático)", 'int x = 3.14', 'Compilado con warning, x=3 (truncado)'),
]
for lang, codigo, resultado in ejemplos_codigo:
    print(f"  {lang:35}: `{codigo}` → {resultado}")

---
## Ejercicio U2-12 (Práctico) — TDS Completa: Declaraciones + Tabla de Símbolos

In [ ]:
# Ejercicio U2-12: TDS completa para declaraciones con tabla de símbolos
# Simula el formalismo del libro: D → T L ;  con TDS L-atribuida

import re

class ProcesadorDeclaraciones:
    """
    Implementa la TDS L-atribuida para declaraciones:
      D → T L ;    { L.tipo_h = T.tipo }
      T → int      { T.tipo = integer, T.ancho = 4 }
      T → float    { T.tipo = float, T.ancho = 8 }
      T → array n of T1 { T.tipo = array(n, T1.tipo); T.ancho = n * T1.ancho }
      L → L1 , id  { L1.tipo_h = L.tipo_h; insertar(id, L.tipo_h) }
      L → id       { insertar(id, L.tipo_h) }
    """

    ANCHOS = {'int': 4, 'float': 8, 'char': 1, 'bool': 1}

    def __init__(self):
        self.tabla = {}
        self.offset = 0   # dirección relativa en memoria

    def insertar(self, nombre, tipo_expr, ancho):
        self.tabla[nombre] = {'tipo': str(tipo_expr), 'dir': self.offset, 'ancho': ancho}
        self.offset += ancho

    def procesar_tipo(self, tokens, pos):
        """Calcula T.tipo y T.ancho (atributos sintetizados)."""
        tok = tokens[pos]
        if tok in ('int', 'float', 'char', 'bool'):
            return tok, self.ANCHOS[tok], pos + 1
        elif tok == 'array':
            n = int(tokens[pos + 1])   # tamaño del array
            # 'of' en tokens[pos+2]
            tipo_elem, ancho_elem, nuevo_pos = self.procesar_tipo(tokens, pos + 3)
            tipo_arr = f"array({n}, {tipo_elem})"
            ancho_arr = n * ancho_elem
            return tipo_arr, ancho_arr, nuevo_pos
        raise SyntaxError(f"Tipo desconocido: {tok}")

    def procesar_declaracion(self, linea):
        tokens = linea.replace(';', '').replace(',', ' ,').split()
        # Extraer el tipo (puede ser compuesto: array N of ...)
        pos = 0
        tipo_val, ancho_tipo, pos = self.procesar_tipo(tokens, pos)

        print(f"  Tipo sintetizado: T.tipo='{tipo_val}', T.ancho={ancho_tipo}")
        print(f"  Tipo heredado → identificadores:")

        # Procesar lista de identificadores con tipo heredado
        ids = [t for t in tokens[pos:] if t != ',']
        for nombre in ids:
            self.insertar(nombre, tipo_val, ancho_tipo)
            print(f"    insertar('{nombre}', tipo='{tipo_val}', dir={self.tabla[nombre]['dir']})")

# Prueba
proc = ProcesadorDeclaraciones()
declaraciones = [
    "int x, y, z;",
    "float precio, iva;",
    "array 10 of int tabla;",
    "array 3 of array 4 of float matriz;",
]
print("TDS L-atribuida — Procesamiento de declaraciones:")
print("=" * 60)
for decl in declaraciones:
    print(f"\n  Declaración: {decl}")
    proc.procesar_declaracion(decl)

print("\nTabla de símbolos final:")
print(f"  {'Nombre':15} | {'Tipo':25} | {'Dir':5} | Ancho")
print("-" * 60)
for nombre, info in proc.tabla.items():
    print(f"  {nombre:15} | {info['tipo']:25} | {info['dir']:5} | {info['ancho']}")

---
## Ejercicio U2-13 (Teórico + Práctico) — Quiz: Comprobación de Tipos

In [ ]:
# Ejercicio U2-13: Quiz de comprobación de tipos

quiz = [
    {
        'pregunta': "¿Cuál es el tipo de: 3 + 4.0?",
        'opciones': ['int', 'float', 'error', 'bool'],
        'correcta': 'float',
        'explicacion': 'int se promueve a float por coerción implícita.',
    },
    {
        'pregunta': "¿Es válida la asignación int x = 3.14; (sin cast)?",
        'opciones': ['Sí, siempre', 'No, error de tipo', 'Sí, con truncamiento', 'Depende del lenguaje'],
        'correcta': 'Depende del lenguaje',
        'explicacion': 'En C: sí (con warning). En Java/Haskell: error. Depende del sistema de tipos.',
    },
    {
        'pregunta': "¿Qué tipo tiene la expresión: x < y siendo x e y enteros?",
        'opciones': ['int', 'float', 'bool', 'error'],
        'correcta': 'bool',
        'explicacion': 'Los operadores de comparación (<, >, ==, !=) siempre producen boolean.',
    },
    {
        'pregunta': "¿Cuál es la estructura para manejar ámbitos anidados en el compilador?",
        'opciones': ['Cola de tablas hash', 'Pila de tablas hash', 'Árbol AVL global', 'Lista enlazada de símbolos'],
        'correcta': 'Pila de tablas hash',
        'explicacion': 'La pila refleja el anidamiento: push al entrar, pop al salir de un bloque.',
    },
    {
        'pregunta': "En equivalencia nominal, ¿son equivalentes T1=array(5,int) y T2=array(5,int) definidos separadamente?",
        'opciones': ['Sí, misma estructura', 'No, nombres distintos', 'Solo si se importan', 'Depende del compilador'],
        'correcta': 'No, nombres distintos',
        'explicacion': 'Equivalencia nominal: solo equivalentes si son el mismo nombre (no la misma estructura).',
    },
]

# Respuestas del estudiante (modifica estas para practicar)
respuestas = ['float', 'Depende del lenguaje', 'bool', 'Pila de tablas hash', 'No, nombres distintos']

print("🧠 QUIZ: Comprobación de Tipos — Unidad 2")
print("=" * 65)
aciertos = 0

for i, (q, r) in enumerate(zip(quiz, respuestas)):
    ok = r == q['correcta']
    if ok: aciertos += 1
    print(f"\n  Pregunta {i+1}: {q['pregunta']}")
    for j, op in enumerate(q['opciones']):
        marcador = '→' if op == r else ' '
        correcto = '✓' if op == q['correcta'] else ' '
        print(f"    {marcador} {correcto} {op}")
    icono = '✅' if ok else '❌'
    print(f"    {icono} {q['explicacion']}")

print(f"\n{'='*65}")
print(f"Resultado: {aciertos}/{len(quiz)} correctas ({100*aciertos//len(quiz)}%)")

---
## Ejercicio U2-14 (Práctico) — Detección de Errores Semánticos

In [ ]:
# Ejercicio U2-14: Sistema de detección y reporte de errores semánticos

from enum import Enum

class TipoError(Enum):
    VARIABLE_NO_DECLARADA  = "Variable no declarada"
    TIPO_INCOMPATIBLE      = "Tipo incompatible"
    DOBLE_DECLARACION      = "Doble declaración"
    ARIDAD_INCORRECTA      = "Número incorrecto de argumentos"
    TIPO_CONDICION         = "Condición no booleana"
    RETORNO_INCORRECTO     = "Tipo de retorno incorrecto"
    INDICE_NO_ENTERO       = "Índice de array no entero"

class ErrorSemantico:
    def __init__(self, tipo, mensaje, linea=None, columna=None):
        self.tipo    = tipo
        self.mensaje = mensaje
        self.linea   = linea
        self.columna = columna

    def __str__(self):
        ubicacion = f" (línea {self.linea})" if self.linea else ""
        return f"[{self.tipo.value}]{ubicacion}: {self.mensaje}"

class GestorErrores:
    def __init__(self):
        self.errores = []
        self.warnings = []

    def agregar(self, tipo, mensaje, linea=None):
        self.errores.append(ErrorSemantico(tipo, mensaje, linea))

    def warning(self, mensaje, linea=None):
        self.warnings.append((mensaje, linea))

    def tiene_errores(self):
        return len(self.errores) > 0

    def reportar(self):
        print(f"\n{'='*60}")
        print(f"REPORTE SEMÁNTICO")
        print(f"{'='*60}")
        if self.errores:
            print(f"\n🔴 ERRORES ({len(self.errores)}):")
            for e in self.errores:
                print(f"  {e}")
        if self.warnings:
            print(f"\n⚠ ADVERTENCIAS ({len(self.warnings)}):")
            for w, l in self.warnings:
                linea = f" (línea {l})" if l else ""
                print(f"  {w}{linea}")
        if not self.errores and not self.warnings:
            print("  ✅ Sin errores ni advertencias")
        print(f"\nEstado: {'❌ COMPILACIÓN FALLIDA' if self.errores else '✅ COMPILACIÓN EXITOSA'}")

# Simulamos el análisis semántico de un programa con errores
gest = GestorErrores()
tabla = {'a': 'int', 'b': 'float', 'flag': 'bool'}

# Línea 1: int a;  → ok
# Línea 2: int a;  → doble declaración!
if 'a' in tabla:
    gest.agregar(TipoError.DOBLE_DECLARACION, "'a' ya fue declarada", linea=2)

# Línea 4: a = b + c;  → c no declarada!
if 'c' not in tabla:
    gest.agregar(TipoError.VARIABLE_NO_DECLARADA, "'c' usada pero no declarada", linea=4)

# Línea 5: a = b;  → float → int, sin cast
if tabla.get('b') == 'float' and tabla.get('a') == 'int':
    gest.warning("Conversión implícita float → int puede perder precisión", linea=5)
    # Podría ser error en lenguaje estricto:
    # gest.agregar(TipoError.TIPO_INCOMPATIBLE, "No se puede asignar float a int sin cast", linea=5)

# Línea 7: if (a + b) then ...  → a+b es float, no bool
tipo_cond = 'float'  # resultado de a + b
if tipo_cond != 'bool':
    gest.agregar(TipoError.TIPO_CONDICION,
                 f"Condición 'if' es de tipo '{tipo_cond}', se esperaba boolean", linea=7)

# Línea 9: tabla[3.14]  → índice float
tipo_indice = 'float'
if tipo_indice != 'int':
    gest.agregar(TipoError.INDICE_NO_ENTERO,
                 f"Índice de array de tipo '{tipo_indice}', se requiere entero", linea=9)

gest.reportar()

---
## Ejercicio U2-15 (Práctico) — Compatibilidad de Tipos en Asignaciones

In [ ]:
# Ejercicio U2-15: Matriz de compatibilidad de tipos

tipos = ['int', 'float', 'char', 'bool', 'string']

# Matriz de compatibilidad para asignación: fila=tipo_variable, col=tipo_expresion
# 'E'=Exacto, 'C'=Coerción implícita, 'X'=Cast explícito, '✗'=Inválido
COMPAT = {
    # var\expr  int    float  char   bool   string
    'int':    ['E',   'X',   'C',   'X',   '✗'],
    'float':  ['C',   'E',   'C',   '✗',   '✗'],
    'char':   ['X',   'X',   'E',   '✗',   '✗'],
    'bool':   ['✗',   '✗',   '✗',   'E',   '✗'],
    'string': ['✗',   '✗',   '✗',   '✗',   'E'],
}

LEYENDA = {'E': '✅ Exacto', 'C': '🔵 Coerción impl.', 'X': '⚠ Cast explícito', '✗': '❌ Inválido'}

print("Matriz de compatibilidad de tipos para asignaciones")
print("(filas = tipo de variable, columnas = tipo de expresión)")
print()
encabezado = f"{'Var \\ Expr':12}" + "".join(f"{t:10}" for t in tipos)
print(encabezado)
print("-" * (12 + 10 * len(tipos)))

for tipo_var in tipos:
    fila = f"  {tipo_var:10}"
    for i, tipo_expr in enumerate(tipos):
        compat = COMPAT[tipo_var][i]
        fila += f"{compat:10}"
    print(fila)

print()
print("Leyenda:")
for k, v in LEYENDA.items():
    print(f"  {k} = {v}")

# Función para consultar compatibilidad
def es_compatible(tipo_var, tipo_expr):
    fila = COMPAT.get(tipo_var, [])
    idx = tipos.index(tipo_expr) if tipo_expr in tipos else -1
    if idx == -1: return '✗', 'Tipo desconocido'
    cod = fila[idx]
    return cod, LEYENDA[cod]

print("\nConsultas de compatibilidad:")
consultas = [
    ('float', 'int', "float f = 5;"),
    ('int', 'float', "int n = 3.14;"),
    ('bool', 'int', "bool b = 1;"),
    ('char', 'int', "char c = 65;"),
    ('string', 'char', "string s = 'a';"),
]
for tv, te, desc in consultas:
    cod, desc_compat = es_compatible(tv, te)
    print(f"  {desc:20} → {desc_compat}")

---
## Ejercicio U2-16 (Práctico) — Verificador Completo con TDS y Tabla de Símbolos

In [ ]:
# Ejercicio U2-16: Verificador semántico completo
# Integra: tabla de símbolos con ámbitos + verificación de tipos

import re

class AnalizadorSemanticoCompleto:
    """Analizador semántico que procesa un mini-lenguaje."""

    TIPOS_BASE = {'int': 4, 'float': 8, 'bool': 1, 'char': 1}
    COERCIONES = {('int', 'float'), ('char', 'int'), ('int', 'int'),
                  ('float', 'float'), ('bool', 'bool'), ('char', 'char')}

    def __init__(self):
        self.pila = [{}]
        self.nivel = 0
        self.errores = []
        self.info = []
        self.offset = 0

    def abrir(self): self.nivel += 1; self.pila.append({})
    def cerrar(self): self.pila.pop(); self.nivel -= 1

    def declarar(self, nombre, tipo, linea=0):
        if nombre in self.pila[-1]:
            self.errores.append(f"L{linea}: '{nombre}' ya declarado en este ámbito")
            return
        ancho = self.TIPOS_BASE.get(tipo, 4)
        self.pila[-1][nombre] = {'tipo': tipo, 'dir': self.offset, 'nivel': self.nivel}
        self.info.append(f"  L{linea}: decl {tipo} {nombre} @ dir={self.offset}")
        self.offset += ancho

    def tipo_var(self, nombre, linea=0):
        for amb in reversed(self.pila):
            if nombre in amb:
                return amb[nombre]['tipo']
        self.errores.append(f"L{linea}: '{nombre}' no declarada")
        return 'error'

    def verificar_asig(self, var, tipo_expr, linea=0):
        tipo_var = self.tipo_var(var, linea)
        ok = (tipo_var, tipo_expr) in self.COERCIONES
        if not ok:
            self.errores.append(f"L{linea}: tipo incompatible: {tipo_expr} → {tipo_var} ({var})")
        else:
            self.info.append(f"  L{linea}: asig {var}:{tipo_var} = expr:{tipo_expr} {'(coerción)' if tipo_var!=tipo_expr else ''}")
        return ok

    def verificar_cond(self, tipo_cond, linea=0):
        ok = tipo_cond == 'bool'
        if not ok:
            self.errores.append(f"L{linea}: condición debe ser bool, es {tipo_cond}")
        return ok

    def reportar(self):
        print("\nAnálisis semántico:")
        for i in self.info: print(i)
        print(f"\nErrores detectados: {len(self.errores)}")
        for e in self.errores: print(f"  ❌ {e}")
        print(f"\n{'✅ OK' if not self.errores else '❌ Falló'}")

# Analizamos este programa:
# int a, b; float x;
# a = 5;         → OK
# x = a;         → coerción int → float, OK
# b = x;         → float → int, ERROR
# if (a) { ... } → a es int, no bool → ERROR

anali = AnalizadorSemanticoCompleto()
anali.declarar('a', 'int',   linea=1)
anali.declarar('b', 'int',   linea=1)
anali.declarar('x', 'float', linea=1)

anali.verificar_asig('a', 'int',   linea=2)   # OK
anali.verificar_asig('x', 'int',   linea=3)   # coerción int→float OK
anali.verificar_asig('b', 'float', linea=4)   # ERROR float→int
anali.verificar_cond('int',        linea=5)   # ERROR: condición no bool

# Bloque anidado
anali.abrir()
anali.declarar('local', 'bool', linea=6)
anali.verificar_cond('bool', linea=7)         # OK
anali.cerrar()

# 'local' ya no existe
anali.tipo_var('local', linea=9)              # ERROR: fuera de scope

anali.reportar()

---
## Ejercicio U2-17 (Teórico) — Comparación de Estrategias de Implementación de Tablas

In [ ]:
# Ejercicio U2-17: Análisis de rendimiento de estructuras para tabla de símbolos

import time, random, string

def benchmark_estructura(nombre_estr, insertar_fn, buscar_fn, n=10000):
    claves = [''.join(random.choices(string.ascii_lowercase, k=8)) for _ in range(n)]

    # Inserción
    t0 = time.perf_counter()
    for k in claves:
        insertar_fn(k, 'int')
    t_ins = (time.perf_counter() - t0) * 1000

    # Búsqueda
    busquedas = random.choices(claves, k=n//2) + ['zzzzzzzz'] * (n//2)
    t0 = time.perf_counter()
    for k in busquedas:
        buscar_fn(k)
    t_bus = (time.perf_counter() - t0) * 1000

    print(f"  {nombre_estr:25} | Inserción: {t_ins:7.2f}ms | Búsqueda: {t_bus:7.2f}ms")

# 1. Tabla hash (dict de Python)
hash_tabla = {}
benchmark_estructura(
    "Tabla Hash (dict)",
    lambda k, v: hash_tabla.update({k: v}),
    lambda k: hash_tabla.get(k)
)

# 2. Lista no ordenada
lista = []
def lista_insertar(k, v): lista.append((k, v))
def lista_buscar(k):
    for clave, val in lista:
        if clave == k: return val
    return None
benchmark_estructura("Lista no ordenada", lista_insertar, lista_buscar, n=2000)

# 3. Árbol binario (usando SortedContainers si disponible, else simulado)
try:
    from sortedcontainers import SortedDict
    arbol = SortedDict()
    benchmark_estructura(
        "Árbol balanceado (SortedDict)",
        lambda k, v: arbol.update({k: v}),
        lambda k: arbol.get(k)
    )
except ImportError:
    print("  Árbol balanceado: SortedContainers no disponible (instalar con pip)")

print("\nConclusión teórica:")
print("  Hash:          O(1) promedio para inserción y búsqueda ← preferido")
print("  Lista:         O(n) búsqueda ← solo para conjuntos pequeños")
print("  Árbol AVL/RN:  O(log n) garantizado ← bueno para conjuntos grandes con garantías")

---
## Ejercicio U2-18 (Práctico) — Verificador de Tipos Recursivo (Tipos Complejos)

In [ ]:
# Ejercicio U2-18: Verificación de tipos en acceso a arrays y punteros

class VerificadorTiposComplejos:
    """Verifica tipos en accesos a arrays y punteros."""

    def __init__(self, tabla):
        self.tabla = tabla
        self.errores = []

    def tipo_de(self, nombre):
        entry = self.tabla.get(nombre)
        if not entry:
            self.errores.append(f"'{nombre}' no declarado")
            return None
        return entry

    def verificar_acceso_array(self, nombre_array, tipo_indice, linea=0):
        """
        E → id[E]:  tipo(id) debe ser array(n, T); índice debe ser int.
        """
        tipo = self.tipo_de(nombre_array)
        if not tipo: return 'error'

        if not isinstance(tipo, TipoArray):
            self.errores.append(f"L{linea}: '{nombre_array}' no es un array (es {tipo})")
            return 'error'

        if tipo_indice not in ('int', 'integer'):
            self.errores.append(f"L{linea}: índice de array debe ser int, es {tipo_indice}")
            return 'error'

        tipo_elem = tipo.tipo_elem
        print(f"  ✅ L{linea}: {nombre_array}[índice:{tipo_indice}] → tipo_elemento: {tipo_elem}")
        return tipo_elem

    def verificar_deref_puntero(self, nombre_ptr, linea=0):
        """
        E → *id:  tipo(id) debe ser pointer(T). Resultado: T.
        """
        tipo = self.tipo_de(nombre_ptr)
        if not tipo: return 'error'

        if not isinstance(tipo, TipoPuntero):
            self.errores.append(f"L{linea}: '{nombre_ptr}' no es puntero (es {tipo})")
            return 'error'

        tipo_apuntado = tipo.tipo_base
        print(f"  ✅ L{linea}: *{nombre_ptr} (ptr→{tipo_apuntado}) → tipo: {tipo_apuntado}")
        return tipo_apuntado

# Tabla de símbolos con tipos complejos
int_t   = TipoBasico('integer')
float_t = TipoBasico('float')

tabla = {
    'arr':    TipoArray(10, int_t),             # array(10, integer)
    'matriz': TipoArray(3, TipoArray(4, float_t)), # array(3, array(4, float))
    'pint':   TipoPuntero(int_t),               # pointer(integer)
    'x':      int_t,                            # integer
    'y':      float_t,                          # float
}

ver = VerificadorTiposComplejos(tabla)

print("Verificación de accesos complejos:")
print("=" * 60)
# Accesos válidos
ver.verificar_acceso_array('arr', 'int', linea=1)          # arr[int] → int
ver.verificar_acceso_array('matriz', 'int', linea=2)       # matriz[int] → array(4, float)
ver.verificar_deref_puntero('pint', linea=3)               # *pint → int

# Errores
ver.verificar_acceso_array('arr', 'float', linea=5)        # índice float → ERROR
ver.verificar_acceso_array('x', 'int', linea=6)            # x no es array → ERROR
ver.verificar_deref_puntero('y', linea=7)                  # y no es puntero → ERROR

print("\n🔴 Errores:")
for e in ver.errores: print(f"  {e}")

---
## Ejercicio U2-19 (Práctico) — Resumen Visual: Verificador de Tipos con Árbol

In [ ]:
# Ejercicio U2-19: Propagación de tipos en un árbol AST con verificación
# Muestra cómo E.tipo se propaga desde las hojas hacia la raíz

class NodoAST:
    def __init__(self, tipo_nodo, hijos=None, valor=None):
        self.tipo_nodo = tipo_nodo
        self.hijos = hijos or []
        self.valor = valor
        self.tipo_sem = None   # ← atributo semántico E.tipo (sintetizado)
        self.error = None

def asignar_tipos(nodo, tabla, errores):
    """Recorre el AST en postorden asignando E.tipo a cada nodo."""
    for hijo in nodo.hijos:
        asignar_tipos(hijo, tabla, errores)

    if nodo.tipo_nodo == 'num_int':
        nodo.tipo_sem = 'int'
    elif nodo.tipo_nodo == 'num_float':
        nodo.tipo_sem = 'float'
    elif nodo.tipo_nodo == 'id':
        nodo.tipo_sem = tabla.get(nodo.valor, 'error')
        if nodo.tipo_sem == 'error':
            errores.append(f"'{nodo.valor}' no declarado")
    elif nodo.tipo_nodo in ('+', '-', '*', '/'):
        t1 = nodo.hijos[0].tipo_sem
        t2 = nodo.hijos[1].tipo_sem
        if t1 in ('int', 'float') and t2 in ('int', 'float'):
            nodo.tipo_sem = 'float' if 'float' in (t1, t2) else 'int'
        else:
            nodo.tipo_sem = 'error'
            errores.append(f"Operación '{nodo.tipo_nodo}' inválida: {t1} y {t2}")
    elif nodo.tipo_nodo == ':=':
        var = nodo.hijos[0].valor
        tipo_var  = tabla.get(var, 'error')
        tipo_expr = nodo.hijos[1].tipo_sem
        ok = (tipo_var == tipo_expr or (tipo_var == 'float' and tipo_expr == 'int'))
        nodo.tipo_sem = tipo_var if ok else 'error'
        if not ok:
            errores.append(f"Asignación inválida: {tipo_expr} → {tipo_var}")

def imprimir_ast_tipado(nodo, nivel=0):
    ind = '  ' * nivel
    tipo_str = f" [{nodo.tipo_sem}]" if nodo.tipo_sem else ""
    val_str  = f" '{nodo.valor}'" if nodo.valor else ""
    icono = '❌' if nodo.tipo_sem == 'error' else ('🔵' if nivel == 0 else '  ')
    print(f"{ind}{icono} {nodo.tipo_nodo}{val_str}{tipo_str}")
    for hijo in nodo.hijos:
        imprimir_ast_tipado(hijo, nivel + 1)

# Construimos AST para: x := a + b * 3.0
# Tabla: a=int, b=int, x=float
tabla = {'a': 'int', 'b': 'int', 'x': 'float'}
errores = []

ast_asig = NodoAST(':=', [
    NodoAST('id', valor='x'),
    NodoAST('+', [
        NodoAST('id', valor='a'),
        NodoAST('*', [
            NodoAST('id', valor='b'),
            NodoAST('num_float', valor=3.0)
        ])
    ])
])

asignar_tipos(ast_asig, tabla, errores)

print("AST con tipos propagados para: x := a + b * 3.0")
print("(E.tipo asignado en postorden — recorrido ascendente)")
print("=" * 55)
imprimir_ast_tipado(ast_asig)
print(f"\nErrores: {errores if errores else 'Ninguno ✅'}")

---
## Ejercicio U2-20 (Integrador) — Mini-Compilador Semántico Completo

Ejercicio final que integra todos los conceptos de la Unidad 2.

In [ ]:
# Ejercicio U2-20: Analizador semántico completo — Integrador Unidad 2

import re

class CompilaS:
    """Mini-compilador semántico completo.
    Soporta: declaraciones, asignaciones, if, while, funciones simples.
    Implementa: tabla de símbolos con ámbitos, verificación de tipos, reporte de errores.
    """

    TIPOS_BASE  = {'int': 4, 'float': 8, 'bool': 1, 'char': 1}
    TIPO_RESULT = {
        ('int',   'int'):   {'arith': 'int',   'comp': 'bool'},
        ('int',   'float'): {'arith': 'float', 'comp': 'bool'},
        ('float', 'int'):   {'arith': 'float', 'comp': 'bool'},
        ('float', 'float'): {'arith': 'float', 'comp': 'bool'},
        ('bool',  'bool'):  {'arith': 'error', 'comp': 'bool'},
    }

    def __init__(self):
        self.pila   = [{}]   # pila de ámbitos
        self.nivel  = 0
        self.errores= []
        self.funciones = {}
        self.stats  = {'decl': 0, 'asig': 0, 'ifs': 0, 'whiles': 0, 'llamadas': 0}

    def abrir(self): self.nivel += 1; self.pila.append({})
    def cerrar(self): self.pila.pop(); self.nivel -= 1

    def err(self, msg, linea=None):
        loc = f"L{linea}: " if linea else ""
        self.errores.append(f"{loc}{msg}")

    def decl(self, nombre, tipo, linea=None):
        if nombre in self.pila[-1]:
            self.err(f"Redeclaración: '{nombre}'", linea); return
        self.pila[-1][nombre] = tipo
        self.stats['decl'] += 1

    def tipo_id(self, nombre, linea=None):
        for amb in reversed(self.pila):
            if nombre in amb: return amb[nombre]
        self.err(f"'{nombre}' no declarada", linea)
        return 'error'

    def tipo_op(self, t1, op, t2, linea=None):
        clave = (t1, t2)
        info = self.TIPO_RESULT.get(clave)
        if not info:
            self.err(f"Operación '{op}' inválida: {t1} y {t2}", linea)
            return 'error'
        cat = 'comp' if op in ('<', '>', '==', '!=', '<=', '>=') else 'arith'
        return info[cat]

    def asig(self, var, tipo_expr, linea=None):
        t_var = self.tipo_id(var, linea)
        compat = (t_var == tipo_expr or (t_var == 'float' and tipo_expr == 'int'))
        if not compat:
            self.err(f"Asignación: {tipo_expr} → {t_var} ({var})", linea)
        self.stats['asig'] += 1

    def verificar_if(self, tipo_cond, linea=None):
        if tipo_cond != 'bool':
            self.err(f"'if': condición debe ser bool, es {tipo_cond}", linea)
        self.stats['ifs'] += 1

    def def_func(self, nombre, params, retorno):
        self.funciones[nombre] = {'params': params, 'retorno': retorno}

    def llamar(self, nombre, tipos_args, linea=None):
        func = self.funciones.get(nombre)
        if not func:
            self.err(f"Función '{nombre}' no definida", linea); return 'error'
        if len(tipos_args) != len(func['params']):
            self.err(f"'{nombre}': aridad incorrecta", linea); return 'error'
        for ta, tp in zip(tipos_args, func['params']):
            if ta != tp and not (ta == 'int' and tp == 'float'):
                self.err(f"'{nombre}': arg {ta} ≠ param {tp}", linea)
        self.stats['llamadas'] += 1
        return func['retorno']

    def reporte(self):
        print("\n" + "="*60)
        print("REPORTE DEL ANALIZADOR SEMÁNTICO")
        print("="*60)
        print(f"  Declaraciones: {self.stats['decl']}")
        print(f"  Asignaciones:  {self.stats['asig']}")
        print(f"  Condicionales: {self.stats['ifs']}")
        print(f"  Llamadas fn:   {self.stats['llamadas']}")
        print(f"  Errores:       {len(self.errores)}")
        if self.errores:
            print("\n  ERRORES:")
            for e in self.errores: print(f"    ❌ {e}")
        else:
            print("  ✅ Sin errores")
        print(f"\n  Estado: {'❌ FALLIDO' if self.errores else '✅ COMPILACIÓN EXITOSA'}")

# ─── Programa de prueba ───────────────────────────────────
# int x, n; float y; bool flag;
# func suma(int, int) → int
# x = 5;           OK
# y = x;           OK (coerción int→float)
# x = y;           ERROR (float→int)
# flag = x < 10;   OK (comparación → bool)
# if (x) { ... }   ERROR (x es int, no bool)
# n = suma(x, 3);  OK
# n = suma(y, 3);  ERROR (arg float ≠ int)

c = CompilaS()
c.def_func('suma', ['int', 'int'], 'int')
c.def_func('prom', ['float', 'float'], 'float')

c.decl('x', 'int', 1); c.decl('n', 'int', 1)
c.decl('y', 'float', 1); c.decl('flag', 'bool', 1)

c.asig('x', 'int',   2)
c.asig('y', 'int',   3)   # OK: coerción
c.asig('x', 'float', 4)   # ERROR

t_cond = c.tipo_op('int', '<', 'int')   # bool
c.asig('flag', t_cond, 5)               # OK
c.verificar_if(c.tipo_id('x'), 6)       # ERROR: x es int
c.verificar_if(c.tipo_id('flag'), 7)    # OK

t_ret = c.llamar('suma', ['int', 'int'], 8)  # OK
c.asig('n', t_ret, 8)

c.llamar('suma', ['float', 'int'], 9)    # ERROR: float ≠ int
c.llamar('noexiste', ['int'], 10)        # ERROR: no definida

# Bloque anidado
c.abrir()
c.decl('local', 'int', 11)
c.asig('local', 'int', 12)  # OK
c.cerrar()
c.tipo_id('local', 13)      # ERROR: fuera de ámbito

c.reporte()

---
## 🎯 Resumen Final

### Unidad 1 — TDS
- Una **TDS** asocia atributos y reglas semánticas a producciones gramaticales.
- **Sintetizados** (↑): fluyen desde las hojas. **Heredados** (↓): fluyen desde el padre.
- **S-atribuida** = solo sintetizados → analizadores ascendentes (LR).
- **L-atribuida** = sintetizados + heredados izquierdos → descendentes (LL).
- El **grafo de dependencias** y la **ordenación topológica** determinan el orden de evaluación.
- Los **esquemas de traducción** son TDS con orden explícito de acciones.

### Unidad 2 — Declaración y Comprobación de Tipos
- La **tabla de símbolos** (hash) almacena tipo, dirección e información de cada identificador.
- La **pila de tablas** gestiona los ámbitos anidados (push/pop al entrar/salir de bloques).
- Las **expresiones de tipo** (árbol) representan tipos complejos (arrays, punteros, funciones).
- **Equivalencia nominal**: mismo nombre. **Estructural**: misma estructura.
- El **verificador de tipos** asigna `E.tipo` en postorden; propaga coerciones implícitas.
- Los errores se clasifican: tipo incompatible, no declarado, condición no booleana, aridad incorrecta.

---
*Cuaderno generado para el curso de Compiladores — Análisis Semántico*